## IMFrag Demo


<img src="figures/IMFrag_Expanded_Graphical_Abstract.png" width="750">

#### This notebook provides a complete set of tools for processing, visualizing, and investigating chemical features in ion mobility-enabled data independent (IM-DIA) workflows. The cells below are designed to be executed in the order they are written; however, in some cases, it may be necessary to iteratively fine-tune certain parameters (i.e., peak fitting thresholds, spectral deconvolution windows) depending on the data in question. 
#### IMFrag is primarily a data visualization tool. Currently, an input target list of MS1 features and paths to the associated data files must be supplied prior to peak picking in the LC and IM dimensions. IMFrag also includes several tools that may facilitate the reconstruction of preecursor-fragment ion relationships; these functionalities are powered by [DEIMoS](https://pubs.acs.org/doi/10.1021/acs.analchem.1c05017), the open-source Python API developed by Pacific Northwest National Laboratory. DEIMoS must be properly installed in order to properly utilize IMFrag. 
#### Contact: ryan97@uw.edu (Ryan Nguyen @ Xu Lab, UW Medicinal Chemistry)


##### (1) Convert Waters .raw LC-IM-MSE (HDMSE) files to .mzML format using Waters2mzML


In [ ]:
# Download the Waters2mzML-1.2.0 converter at https://github.com/AnP311/Waters2mzML/releases (Written and developed by Anja Prisching)

# This converter should be run directly as a stand-alone utility
# Navigate to the Waters2mzML-1.2.0 folder and place COPIES of the .raw files in the raw_files folder (NEVER place original files here!)
# Launch the application and type "y" to the question regarding centroiding; Waters HDMSE files always contain continuum data, by default
# Converted mzML files will appear in the mzML_files folder
# NOTE: If pre-centroided (MassLynx) LC-IM-MS files are provided, the drift time dimension will NOT be retained!
# NOTE: Running the .exe will ONLY work on LC-IM-MS files containing a LockMass (REFERENCE) function; all higher functions will be deleted otherwise
# To safely convert HDMSE files WITHOUT a REFERENCE function, refer to the instructions directly below; otherwise, proceed to the subsequent cells


##### READ BELOW CAREFULLY to handle LC-IM-HDMSE files that DO NOT contain a LockMass (REFERENCE) function


In [ ]:
# Waters2mzML is encoded to identify and then delete the REFERENCE function; all higher functions are ignored
# To bypass this behavior without a permanent patch, it is possible to simply paste a decoy REFERENCE function into the _extern file
# The specified function number should be exactly one integer higher than the highest REAL function number
# For example, in a standard HDMSE experiment containing both low and high energy scans, the _extern file will contain two functions:
# Function 1 = low energy (no collision energy), and Function 2 = high energy (collision energy, either fixed or ramped)
# In this specific case, simply open the _extern file and paste the following text directly below Function 2:

Function Parameters - Function 3 - REFERENCE
Scan Time (sec)					1.000
Interscan Time (sec)				0.100
Start Mass					50.0
End Mass					1200.0
Start Time (mins)				0.20
End Time (mins)					1.15
Data Format					Continuum
Analyser					Resolution Mode
ADC Sample Frequency (GHz)			3.0
ADC Pusher Frequency ( s)			66.0
ADC Pusher Width ( s)				1.50
Use Tune Page Cone Voltage			YES
Using Auto Trap Collision Energy (eV)		4.000000
Using Auto Transfer Collision Energy (eV)	2.000000
Sensitivity					Normal
Dynamic Range					Normal
Save Collapsed Retention Time Data		Yes
Use Rule File Filtering				No
FragmentationMode				CID
[LOCK SPRAY]
LockSprayType:					Do not apply correction
LockSpray Reference Compound Name:		ResSP_LE
Scan Time (sec)					1.00
Interval					60
EDC Mass					0.0000
Mass Window +/-					0.5
Scans to Average				2.0
Reference Cone Voltage				30.0
Trap Collision Energy (eV)			4.0
LocksprayDRESetting				63.7200

# Note: the experimental details should be irrelevant; Waters2mzML will read the REFERENCE line and continue


##### (2) Convert all output .txt files to .mzML files for downstream processing


In [ ]:
# Recommended to activate "IMFrag" environment here; otherwise, tested running Python 3.10.9

# Input libraries
from pathlib import Path
import shutil

# Define path to folder containing .txt files (the output files of Waters2mzML-1.2.0 when run in batch mode)
# Recommended to move or copy the output .txt files from Waters2mzML-1.2.0 to a new folder (e.g., "mzML Data Files")
folder = Path("mzML Data Files")  # Edit path, if necessary
txt_files = folder.glob("*.txt")

for txt in txt_files:
    mzml_path = txt.with_suffix(".mzML")
    shutil.copy(txt, mzml_path)
    print(f"Copied: {txt.name} → {mzml_path.name}")

print("Converted files to mzML.")


##### (3) Import DEIMoS (REQUIRED for data conversion and downstream spectral deconvolution tools)


In [ ]:
# NOTE: Activate the "imfrag" environment, if not already activated

# Install DEIMoS, if not already installed
# For detailed instructions, refer to the DEIMoS documentation here: https://deimos.readthedocs.io/en/latest/getting_started/installation.html
import deimos

print("DEIMoS version:", getattr(deimos, "__version__", "unknown"))


##### (4) Load the directory of .mzML files in DEIMoS and save as .h5 


In [ ]:
# Input libraries
from pathlib import Path
import traceback

# Define path to folder containing .mzML files
INPUT_DIR = Path("mzML Data Files")  # Edit path, if necessary

# Converted .h5 files will be saved to this folder
# Default name is "Hierarchical Data Files"
OUTPUT_DIR = Path("Hierarchical Data Files")

# Create the output directory folder, if not already created
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


##### (5) Run the cell below to convert the folder of .mzML files to .h5


In [ ]:
OVERWRITE = False

# Define accessions confirmed from cell above
ACCESSION_MAP = {
    "retention_time": "MS:1000016",   # Scan start time (minutes)
    "drift_time": "MS:1002476",       # Ion mobility drift time (ms or seconds; depends on writer)
}

# Convert one .mzML to .h5 with keys "ms1" and "ms2"
# Return a small dictionary of counts and the output path
def convert_one_mzml(mzml_path: Path, out_dir: Path, overwrite: bool = False) -> dict:
    out_dir.mkdir(parents=True, exist_ok=True)
    h5_path = out_dir / (mzml_path.stem + ".h5")
    if h5_path.exists() and not overwrite:
        return {"path": h5_path, "status": "skipped (exists)", "ms1_rows": None, "ms2_rows": None}

    # Load both MS levels
    data = deimos.load(str(mzml_path), accession=ACCESSION_MAP)

    # Basic check
    if "ms1" not in data or "ms2" not in data:

        # still try to save whatever exists for debugging
        if "ms1" in data:
            deimos.save(str(h5_path), data["ms1"], key="ms1", mode="w")
        if "ms2" in data:
            deimos.save(str(h5_path), data["ms2"], key="ms2", mode="a")
        have = ",".join(sorted(data.keys()))
        return {"path": h5_path, "status": f"partial (had: {have})",
                "ms1_rows": len(data.get("ms1", [])),
                "ms2_rows": len(data.get("ms2", []))}

    # Save MS1 then MS2
    deimos.save(str(h5_path), data["ms1"], key="ms1", mode="w")
    deimos.save(str(h5_path), data["ms2"], key="ms2", mode="a")

    return {"path": h5_path, "status": "converted", "ms1_rows": len(data["ms1"]), "ms2_rows": len(data["ms2"])}

def convert_folder(mzml_dir: Path, out_dir: Path, overwrite: bool = False):
    mzml_dir = Path(mzml_dir).resolve()
    out_dir = mzml_dir.parent / out_dir
    out_dir.mkdir(parents=True, exist_ok=True)

    # Collect *.mzML and *.mzml
    files = list(mzml_dir.glob("*.mzML")) + list(mzml_dir.glob("*.mzml"))
    files = list({f.resolve(): f for f in files}.values())
    files.sort()
    if not files:
        print(f"No .mzML files found in: {mzml_dir}")
        return

    print(f"Converting {len(files)} file(s) from:\n  {mzml_dir}\n→ HDF5 to:\n  {out_dir}\n"
          f"(overwrite={'ON' if overwrite else 'OFF'})\n")

    successes = skips = failures = 0
    for i, f in enumerate(files, 1):
        try:
            info = convert_one_mzml(f, out_dir, overwrite)
            status = info["status"]
            if status.startswith("converted"):
                successes += 1
                print(f"[{i:>3}/{len(files)}] converted   → {info['path'].name}   "
                      f"(ms1={info['ms1_rows']:,}, ms2={info['ms2_rows']:,})")
            elif status.startswith("skipped"):
                skips += 1
                print(f"[{i:>3}/{len(files)}] skipped → {info['path'].name}")
            else:
                successes += 1
                print(f"[{i:>3}/{len(files)}] partial → {info['path'].name}   status={status}")
        except Exception as e:
            failures += 1
            print(f"[{i:>3}/{len(files)}] FAILED  → {f.name}\n   {e}")
            traceback.print_exc(limit=1)

    print("\nSummary:")
    print(f"  converted: {successes}")
    print(f"  skipped:   {skips}")
    print(f"  failed:    {failures}")

# Run
convert_folder(INPUT_DIR, OUTPUT_DIR, overwrite=OVERWRITE)


##### OPTIONAL: Edit and run the two cells below to visually inspect a selected .h5 file


In [ ]:
# Define path to .h5 file to be visually inspected
h5_path = Path(r"C:\Users\ryann\Desktop\ToxCast Metabolism\Hierarchical Data Files\Plasma_203_HDMSE.h5")  # Edit path

In [ ]:
from pathlib import Path
import pandas as pd

def inspect_h5(h5_path: Path, n: int = 5):
    print("FILE:", h5_path)
    with pd.HDFStore(h5_path, mode="r") as store:
        keys = store.keys()
        print("\nKeys:", keys)

        for k in keys:
            df = store.select(k) # Loads the whole file
            print(f"\n=== {k} ===")
            print("shape:", df.shape)
            print("columns:", list(df.columns))
            print("\ndtypes:")
            print(df.dtypes)
            print(f"\nhead({n}):")
            display(df.head(n))

# Edit "n" depending on how many files to inspect
inspect_h5(h5_path, n=10) 

##### (6) Run the cell below to load .h5 files into pandas DataFrames


In [ ]:
import h5py
import numpy as np
import pandas as pd
from pathlib import Path

def _pick_first_dataset(group: h5py.Group) -> h5py.Dataset:
    """Pick a likely 'main' dataset inside a group."""
    ds_paths: list[str] = []

    def visitor(name, obj):
        if isinstance(obj, h5py.Dataset):
            ds_paths.append(name)

    group.visititems(visitor)
    if not ds_paths:
        raise TypeError("Group contains no datasets.")
    
    # Prefer common names
    preferred = sorted(
        ds_paths,
        key=lambda p: (
            0 if any(k in p.lower() for k in ["data", "table", "ms1", "ms2"]) else 1,
            len(p),
        ),
    )
    return group[preferred[0]]


def _get_columns_for_matrix(node) -> list[str] | None:
    """
    Try to recover column names for a 2D matrix table.
    """

    # If node is a dataset, check its attributes
    if isinstance(node, h5py.Dataset):
        for attr in ("columns", "colnames", "column_names"):
            if attr in node.attrs:
                cols = node.attrs[attr]
                return [c.decode() if isinstance(c, (bytes, bytearray)) else str(c) for c in cols]
        
        # If dataset has a parent group, check attributes
        parent = node.parent
        if isinstance(parent, h5py.Group):
            if "columns" in parent:
                cols = parent["columns"][:]
                return [c.decode() if isinstance(c, (bytes, bytearray)) else str(c) for c in cols]
            for attr in ("columns", "colnames", "column_names"):
                if attr in parent.attrs:
                    cols = parent.attrs[attr]
                    return [c.decode() if isinstance(c, (bytes, bytearray)) else str(c) for c in cols]
    
    # If node is a group, check it directly
    if isinstance(node, h5py.Group):
        if "columns" in node:
            cols = node["columns"][:]
            return [c.decode() if isinstance(c, (bytes, bytearray)) else str(c) for c in cols]
        for attr in ("columns", "colnames", "column_names"):
            if attr in node.attrs:
                cols = node.attrs[attr]
                return [c.decode() if isinstance(c, (bytes, bytearray)) else str(c) for c in cols]
    return None

def _node_to_dataframe(node, fallback_prefix: str) -> pd.DataFrame:
    if isinstance(node, h5py.Group):
        ds = _pick_first_dataset(node)
        arr = ds[:]
        cols = _get_columns_for_matrix(node) or _get_columns_for_matrix(ds)
    elif isinstance(node, h5py.Dataset):
        ds = node
        arr = ds[:]
        cols = _get_columns_for_matrix(ds)
    else:
        raise TypeError(f"Unsupported node type: {type(node)}")

    # Structured dtype
    if hasattr(arr, "dtype") and arr.dtype.names is not None:
        data: dict[str, np.ndarray] = {}
        for name in arr.dtype.names:
            v = np.asarray(arr[name])

            # Normal 1D column
            if v.ndim == 1:
                data[name] = v
            
            # Subarray field: expand into multiple 1D columns
            elif v.ndim == 2:
                for j in range(v.shape[1]):
                    data[f"{name}_{j}"] = v[:, j]
            else:
                raise TypeError(f"Field '{name}' has unexpected ndim={v.ndim} and shape={v.shape}")
        df = pd.DataFrame(data)
        if cols is not None and len(cols) == df.shape[1]:
            df.columns = cols
        return df

    arr = np.asarray(arr)
    if arr.ndim != 2:
        raise TypeError(f"Expected 2D matrix or structured table, got ndim={arr.ndim}, shape={arr.shape}")
    n_rows, n_cols = arr.shape

    # If we expaded a single 2D field and user provided true column names, apply
    if cols is None or len(cols) != n_cols:
        cols = [f"{fallback_prefix}_{i}" for i in range(n_cols)]
    return pd.DataFrame(arr, columns=cols)


_MS1_COLS = ["function", "process", "scan", "retention_time", "drift_time", "mz", "intensity"]
_MS2_COLS = _MS1_COLS + ["precursor_mz"]


def _normalize_columns(df: pd.DataFrame, expect: str) -> pd.DataFrame:
    if {f"{expect}_{i}" for i in range(7 if expect == "ms1" else 8)}.issubset(df.columns):
        df.columns = _MS1_COLS if expect == "ms1" else _MS2_COLS
        return df
    if "values_block_0_0" in df.columns:
        df = df.drop(columns=["index"], errors="ignore").copy()
        n_block = 7 if expect == "ms1" else 8
        rename = {f"values_block_0_{i}": (_MS1_COLS if expect == "ms1" else _MS2_COLS)[i] for i in range(n_block)}
        df = df.rename(columns=rename)
    expected = _MS1_COLS if expect == "ms1" else _MS2_COLS
    if all(c in df.columns for c in expected):
        return df[expected]
    return df


def load_ms_tables(h5_path: Path, verbose: bool = True) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load /ms1 and /ms2 from a DEIMoS-written .h5 into two pandas DataFrames."""
    h5_path = Path(h5_path)

    # Try pandas read_hdf first 
    try:
        ms1_df = pd.read_hdf(h5_path, key="/ms1")
        ms2_df = pd.read_hdf(h5_path, key="/ms2")
        ms1_df = _normalize_columns(ms1_df, "ms1")
        ms2_df = _normalize_columns(ms2_df, "ms2")
        if verbose:
            print(f"[H5] {h5_path.name} (pandas)")
            print("  ms1_df:", ms1_df.shape, list(ms1_df.columns))
            print("  ms2_df:", ms2_df.shape, list(ms2_df.columns))
        return ms1_df, ms2_df
    except Exception:
        pass

    # Fallback
    with h5py.File(h5_path, "r") as h5:
        if "/ms1" not in h5 or "/ms2" not in h5:
            raise KeyError(f"Expected keys '/ms1' and '/ms2' in {h5_path}")
        ms1_df = _node_to_dataframe(h5["/ms1"], "ms1")
        ms2_df = _node_to_dataframe(h5["/ms2"], "ms2")
    ms1_df = _normalize_columns(ms1_df, "ms1")
    ms2_df = _normalize_columns(ms2_df, "ms2")
    if verbose:
        print(f"[H5] {h5_path.name} (h5py)")
        print("  ms1_df:", ms1_df.shape, list(ms1_df.columns))
        print("  ms2_df:", ms2_df.shape, list(ms2_df.columns))
    return ms1_df, ms2_df


# Run for all .h5 files in OUTPUT_DIR
H5_DIR = Path(OUTPUT_DIR)
h5_files = sorted(H5_DIR.glob("*.h5"))

print(f"Found {len(h5_files)} h5 file(s)")

ms_data: dict[str, dict] = {}
for h5_path in h5_files:
    ms1_df, ms2_df = load_ms_tables(h5_path, verbose=True)
    ms_data[h5_path.stem] = {"ms1": ms1_df, "ms2": ms2_df, "path": h5_path}

##### (7) Edit the cell below to load in MS1 target information from Excel spreadsheet


In [ ]:
# The spreadsheet should contain the following headers and information for each MS1 feature of interest
# NOTE: All columns OTHER than Target m/z and File Name are purely for internal data tracking in the output .txt files; the values stored in these columns DO NOT influence the output
# Feature Name (REQUIRED)  --  Name of the MS1 feature that MS2 features will be assigned to; may be as simple as "Candidate_1", "Candidate_2"... etc. when the feature name is unknown
# Target m/z   (REQUIRED)  --  Target m/z value of the MS1 feature; recommended to use the m/z value(s) of interest that are observed in the raw data
# File Name    (REQUIRED)  --  Associated HD5 (.h5) file; note that the cells above are used for converting .raw (Waters) --> .mzML (Waters2mzML) --> .h5

# Define path to Excel spreadsheet containing MS1 target list
TARGETS_XLSX = Path("Plasma_203_IMFrag.xlsx")


##### Run the cell below to load in targets


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# Confirm input target list contains required column headers
REQUIRED_COLS = ["Feature Name", "Target m/z", "File Name"]

def read_targets_excel(path: Path) -> pd.DataFrame:
    df = pd.read_excel(path).copy()
    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    df["Target m/z"] = pd.to_numeric(df["Target m/z"], errors="coerce")
    if df["Target m/z"].isna().any():
        bad = df[df["Target m/z"].isna()][REQUIRED_COLS]
        raise ValueError(f"Some Target m/z could not be parsed:\n{bad.to_string(index=False)}")
    return df[REQUIRED_COLS].copy()

def resolve_h5_path(file_cell, h5_dir: Path) -> Path:
    p = Path(str(file_cell))
    if p.suffix.lower() != ".h5":
        p = p.with_suffix(".h5")
    return p if p.is_absolute() else (h5_dir / p)

targets_df = read_targets_excel(TARGETS_XLSX)
print(f"Loaded {len(targets_df)} targets")
display(targets_df.head())


##### (8) Investigate each MS1 feature to construct extracted ion chromatograms (EICs) and mobilograms
##### Edit the cell below to tune parameters for LC peak picking


In [ ]:
# Tolerance in ppm around the target m/z used to filter ions in the MS1 and MS2 tables
MZ_PPM = 25.0

# Retention time bin size in min; used to build the MS1 extracted ion chromatogram (EIC)
RT_BIN_MIN = 0.01     # Smaller bins = higher resolution

# Minimum peak height threshold relative to the maximum signal in the trace
MIN_REL_HEIGHT = 0.5  # Example: 0.5 = only peaks with height >= 50% of max will be considered

# Minimum prominence threshold relative to the maximum signal in the trace
PROMINENCE_REL = 0.05 # Edit to include/exclude shouldering peaks

# Relative height for peak widths; used to define peak boundaries
LC_REL_HEIGHT = 0.5   # Example: 0.5 = approximate FWHM-like boundaries

# Savitzky-Golay smoothing parameters (must be odd; code will auto-fix if even)
SGOLAY_WINDOW = 9     # Recommended: 5-15
SGOLAY_POLY = 2       # Recommended: 2-3

# Drift time bin size in ms; used to construct ATDs
DT_BIN_MS = None      # Recommended to keep None to mitigate artifacts due to peak picking

# If True and MS2 table includes precursor_mz, also extract an MS2 ATD restricted to scans
# whose precursor_mz matches the target m/z
ALSO_EXTRACT_MS2_BY_PRECURSOR = True

# Printing and debugging
VERBOSE = True

##### Run the cell below to construct EICs and mobilograms for all MS1 targets

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.signal import find_peaks, peak_widths, savgol_filter

def ppm_window(mz: float, mz_ppm: float) -> tuple[float, float]:
    mz = float(mz)
    ppm = float(mz_ppm) * 1e-6
    return mz * (1 - ppm), mz * (1 + ppm)

def build_ms1_eic(ms1_df: pd.DataFrame,
                  target_mz: float,
                  mz_ppm: float,
                  rt_bin_min: float = 0.01) -> tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    """Build a uniformly binned MS1 EIC at target_mz +/- mz_ppm. Returns (rt_centers, eic, sub)."""
    lo, hi = ppm_window(target_mz, mz_ppm)
    sub = ms1_df[(ms1_df["mz"] >= lo) & (ms1_df["mz"] <= hi)].copy()
    if sub.empty:
        return np.array([]), np.array([]), sub

    rt = sub["retention_time"].to_numpy(float)
    inten = sub["intensity"].to_numpy(float)

    rt0, rt1 = float(np.min(rt)), float(np.max(rt))
    if rt1 <= rt0:
        return np.array([rt0]), np.array([float(np.sum(inten))]), sub

    nbins = int(np.ceil((rt1 - rt0) / rt_bin_min)) + 1
    edges = rt0 + np.arange(nbins + 1) * rt_bin_min
    idx = np.clip(np.digitize(rt, edges) - 1, 0, nbins - 1)
    eic = np.bincount(idx, weights=inten, minlength=nbins).astype(float)
    centers = edges[:-1] + rt_bin_min / 2.0
    return centers.astype(float), eic, sub

def find_lc_peak_windows(rt_centers, y,
                         min_rel_height: float = 0.01,
                         prominence_rel: float = 0.02,
                         lc_rel_height: float = 0.5,
                         sgolay_window: int = 9,
                         sgolay_poly: int = 2):
    """Detect LC peaks. Returns list of (rt_left, rt_right, rt_apex, apex_height) sorted by apex height desc."""
    rt_centers = np.asarray(rt_centers, float)
    y = np.asarray(y, float)

    if len(y) == 0 or np.nanmax(y) <= 0:
        return []

    y_pick = y.copy()
    if sgolay_window is not None and len(y) >= max(5, int(sgolay_window)):
        w = int(sgolay_window)
        if w % 2 == 0:
            w += 1
        w = min(w, len(y) if len(y) % 2 == 1 else len(y) - 1)
        w = max(w, 5)
        if w > sgolay_poly:
            y_pick = savgol_filter(y, window_length=w, polyorder=int(sgolay_poly), mode="interp")

    ymax = float(np.nanmax(y_pick))
    peaks, _ = find_peaks(y_pick,
                          height=ymax * float(min_rel_height),
                          prominence=ymax * float(prominence_rel))
    if len(peaks) == 0:
        return []

    w = peak_widths(y_pick, peaks, rel_height=float(lc_rel_height))
    left_ips, right_ips = w[2], w[3]

    out = []
    for p, li, ri in zip(peaks, left_ips, right_ips):
        li_i = max(int(np.floor(li)), 0)
        ri_i = min(int(np.ceil(ri)), len(rt_centers) - 1)
        out.append((float(rt_centers[li_i]),
                    float(rt_centers[ri_i]),
                    float(rt_centers[p]),
                    float(y_pick[p])))
    out.sort(key=lambda x: x[3], reverse=True)
    return out

def extract_atd(df: pd.DataFrame,
                mz_col: str,
                target_mz: float,
                mz_ppm: float,
                rt_left: float,
                rt_right: float,
                dt_bin_ms=None) -> tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    """
    Extract an ATD over the given m/z + RT window.
    """
    lo, hi = ppm_window(target_mz, mz_ppm)
    sub = df[
        (df[mz_col] >= lo) & (df[mz_col] <= hi) &
        (df["retention_time"] >= float(rt_left)) & (df["retention_time"] <= float(rt_right))
    ].copy()

    if sub.empty:
        return np.array([]), np.array([]), sub

    dt = sub["drift_time"].to_numpy(float)
    inten = sub["intensity"].to_numpy(float)

    if dt_bin_ms is None:
        u = np.unique(dt)
        if len(u) == 0:
            return np.array([]), np.array([]), sub
        inv = np.searchsorted(u, dt)
        atd = np.bincount(inv, weights=inten, minlength=len(u)).astype(float)
        centers = u.astype(float)
    else:
        dt0, dt1 = float(np.min(dt)), float(np.max(dt))
        if dt1 <= dt0:
            return np.array([dt0]), np.array([float(np.sum(inten))]), sub
        dt_bin_ms = float(dt_bin_ms)
        nbins = int(np.ceil((dt1 - dt0) / dt_bin_ms)) + 1
        edges = dt0 + np.arange(nbins + 1) * dt_bin_ms
        idx = np.clip(np.digitize(dt, edges) - 1, 0, nbins - 1)
        atd = np.bincount(idx, weights=inten, minlength=nbins).astype(float)
        centers = edges[:-1] + dt_bin_ms / 2.0

    # Trim contiguous zero leading/trailing rows
    if atd.size > 0 and float(np.max(atd)) > 0:
        nz = atd > 0
        first = int(np.argmax(nz))
        last = int(len(nz) - 1 - np.argmax(nz[::-1]))
        centers = centers[first:last + 1]
        atd = atd[first:last + 1]

    return centers, atd, sub

def extract_all_instances(input_sheet: Path,
                          h5_dir: Path,
                          mz_ppm: float = 10.0,
                          rt_bin_min: float = 0.01,
                          min_rel_height: float = 0.01,
                          prominence_rel: float = 0.02,
                          lc_rel_height: float = 0.5,
                          sgolay_window: int = 9,
                          sgolay_poly: int = 2,
                          dt_bin_ms=None,
                          also_extract_ms2_by_precursor: bool = True,
                          verbose: bool = True) -> tuple[pd.DataFrame, dict]:
    targets = read_targets_excel(input_sheet)

    cache: dict[Path, tuple[pd.DataFrame, pd.DataFrame]] = {}
    instances_rows: list[dict] = []
    atd_store: dict[tuple[int, int], dict] = {}

    for row_index, row in targets.iterrows():
        feature_name = str(row["Feature Name"])
        target_mz = float(row["Target m/z"])
        file_name = str(row["File Name"])
        h5_path = resolve_h5_path(file_name, h5_dir)

        if verbose:
            print(f"\n[{row_index}] {feature_name} | m/z: {target_mz:.6f} | {h5_path.name}")

        if h5_path not in cache:
            cache[h5_path] = load_ms_tables(h5_path, verbose=False)
        ms1_df, ms2_df = cache[h5_path]

        rt_centers, eic, _ = build_ms1_eic(ms1_df, target_mz, mz_ppm, rt_bin_min)
        if len(eic) == 0 or np.nanmax(eic) <= 0:
            if verbose:
                print("   No EIC signal; skipping")
            continue

        lc_windows = find_lc_peak_windows(
            rt_centers, eic,
            min_rel_height=min_rel_height,
            prominence_rel=prominence_rel,
            lc_rel_height=lc_rel_height,
            sgolay_window=sgolay_window,
            sgolay_poly=sgolay_poly,
        )

        if verbose:
            print(f"    LC peaks: {len(lc_windows)}")

        for lc_peak_index, (rt_left, rt_right, rt_apex, apex_height) in enumerate(lc_windows, start=1):
            ms1_dt, ms1_atd, _ = extract_atd(ms1_df, "mz", target_mz, mz_ppm, rt_left, rt_right, dt_bin_ms)
            ms2_dt, ms2_atd, _ = extract_atd(ms2_df, "mz", target_mz, mz_ppm, rt_left, rt_right, dt_bin_ms)

            store_key = (row_index, lc_peak_index)
            atd_store[store_key] = {
                "ms1_dt": ms1_dt, "ms1_atd": ms1_atd,
                "ms2_same_mz_dt": ms2_dt, "ms2_same_mz_atd": ms2_atd,
            }

            if also_extract_ms2_by_precursor and ("precursor_mz" in ms2_df.columns):
                lo_p, hi_p = ppm_window(target_mz, mz_ppm)
                ms2_prec = ms2_df[(ms2_df["precursor_mz"] >= lo_p) & (ms2_df["precursor_mz"] <= hi_p)].copy()
                ms2p_dt, ms2p_atd, _ = extract_atd(ms2_prec, "mz", target_mz, mz_ppm, rt_left, rt_right, dt_bin_ms)
                atd_store[store_key]["ms2_by_precursor_dt"] = ms2p_dt
                atd_store[store_key]["ms2_by_precursor_atd"] = ms2p_atd

            instances_rows.append({
                "row_index": row_index,
                "lc_peak_index": lc_peak_index,
                "feature_name": feature_name,
                "target_mz": target_mz,
                "file_name": file_name,
                "h5_path": str(h5_path),
                "rt_left": rt_left,
                "rt_right": rt_right,
                "rt_apex": rt_apex,
                "rt_apex_intensity": apex_height,
                "ms1_atd_bins": int(len(ms1_dt)),
                "ms2_same_mz_atd_bins": int(len(ms2_dt)),
            })

            if verbose:
                print(f"    [LC {lc_peak_index}] RT bounds: {rt_left:.4f}-{rt_right:.4f} min "
                      f"| MS1 bins: {len(ms1_dt)} | MS2 (same m/z) bins: {len(ms2_dt)}")

    return pd.DataFrame(instances_rows), atd_store

instances_df, atd_store = extract_all_instances(
    input_sheet=TARGETS_XLSX,
    h5_dir=H5_DIR,
    mz_ppm=MZ_PPM,
    rt_bin_min=RT_BIN_MIN,
    min_rel_height=MIN_REL_HEIGHT,
    prominence_rel=PROMINENCE_REL,
    lc_rel_height=LC_REL_HEIGHT,
    sgolay_window=SGOLAY_WINDOW,
    sgolay_poly=SGOLAY_POLY,
    dt_bin_ms=DT_BIN_MS,
    also_extract_ms2_by_precursor=ALSO_EXTRACT_MS2_BY_PRECURSOR,
    verbose=VERBOSE,
)

display(instances_df.head(20))

##### Edit the cell below to visualize extracted mobilograms for a selected MS1 feature


In [ ]:
# Enter the name of the feature (EXACTLY as listed in the Excel spreadsheet) to be plotted and visualized
FEATURE_NAME = "Plasma_144"

# Set to None to plot all peaks for this feature
LC_PEAK_INDEX = None

# OPTIONAL: If multiple features share the same name, specify which to plot with the row_index (above)
ROW_INDEX = 0


##### Run the cell below to generate mobilogram plots


In [ ]:
import matplotlib.pyplot as plt
from matplotlib import rcParams
import numpy as np

rcParams["font.family"] = "Arial"


def plot_ms1_ms2_atds_single_figure(instances_df, atd_store, feature_name,
                                    lc_peak_index=None, row_index=None):
    df = instances_df[instances_df["feature_name"].astype(str) == str(feature_name)].copy()
    if df.empty:
        raise ValueError(f"No instances found for feature_name: '{feature_name}'")
    if row_index is not None:
        df = df[df["row_index"] == row_index].copy()
        if df.empty:
            raise ValueError(f"No instances for feature_name='{feature_name}' with row_index={row_index}")
    if lc_peak_index is not None:
        df = df[df["lc_peak_index"] == lc_peak_index].copy()
        if df.empty:
            raise ValueError(f"No instances for feature_name='{feature_name}' lc_peak_index={lc_peak_index}")

    df = df.sort_values(["row_index", "lc_peak_index"]).reset_index(drop=True)

    for _, inst in df.iterrows():
        key = (int(inst["row_index"]), int(inst["lc_peak_index"]))
        if key not in atd_store:
            print(f"Missing ATD data for key: {key}")
            continue
        data = atd_store[key]

        fig, (ax1, ax2, ax3) = plt.subplots(nrows=3, ncols=1, figsize=(6.4, 8.6), sharex=True)

        # MS1 ATD
        if len(data.get("ms1_dt", [])) > 0:
            ax1.plot(data["ms1_dt"], data["ms1_atd"], lw=3, color="black", label="MS1")
        else:
            ax1.text(0.5, 0.5, "No MS1 ATD data", ha="center", va="center", transform=ax1.transAxes)
        ax1.set_ylabel("Intensity", fontsize=12, fontweight="bold", fontname="Arial")
        ax1.set_title(
            f"{inst['feature_name']}\n"
            f"{inst['target_mz']:.4f} m/z\n"
            f"LC peak {int(inst['lc_peak_index'])} | RT Bounds: {inst['rt_left']:.2f} - {inst['rt_right']:.2f} min\n\n"
            f"Extracted Arrival Time Distributions",
            fontsize=14, fontweight="bold",
        )

        # MS2 ATD
        if len(data.get("ms2_same_mz_dt", [])) > 0:
            ax2.plot(data["ms2_same_mz_dt"], data["ms2_same_mz_atd"], lw=3, color="tab:blue", label="MS2")
        else:
            ax2.text(0.5, 0.5, "No MS2 (same m/z) ATD data", ha="center", va="center", transform=ax2.transAxes)
        ax2.set_ylabel("Intensity", fontsize=12, fontweight="bold", fontname="Arial")

        # Overlay
        ms1_plotted = ms2_plotted = False
        if len(data.get("ms1_dt", [])) > 0:
            ax3.plot(data["ms1_dt"], data["ms1_atd"], lw=3, color="black", label="MS1")
            ms1_plotted = True
        if len(data.get("ms2_same_mz_dt", [])) > 0:
            ax3.plot(data["ms2_same_mz_dt"], data["ms2_same_mz_atd"], lw=2, color="tab:blue", label="MS2")
            ms2_plotted = True
        if not (ms1_plotted or ms2_plotted):
            ax3.text(0.5, 0.5, "No ATD data to overlay", ha="center", va="center", transform=ax3.transAxes)
        ax3.set_xlabel("Drift Time (ms)", fontsize=12, fontweight="bold", fontname="Arial")
        ax3.set_ylabel("Intensity", fontsize=12, fontweight="bold", fontname="Arial")

        for ax in (ax1, ax2, ax3):
            ax.tick_params(axis="both", which="major", labelsize=10, width=2)
            for label in ax.get_xticklabels() + ax.get_yticklabels():
                label.set_fontweight("bold")
                label.set_fontsize(12)
            for side in ("top", "right", "left", "bottom"):
                ax.spines[side].set_linewidth(3)
            if ax.get_lines():
                ax.legend(loc="best", frameon=True, fontsize=12, edgecolor="black", facecolor="white")
                y_max = max(np.max(line.get_ydata()) for line in ax.get_lines())
                ax.set_ylim(0, y_max * 1.2)
            else:
                ax.set_ylim(0, 1)

        plt.tight_layout()
        plt.show()


plot_ms1_ms2_atds_single_figure(instances_df=instances_df,
                                atd_store=atd_store,
                                feature_name=FEATURE_NAME,
                                lc_peak_index=LC_PEAK_INDEX,
                                row_index=ROW_INDEX)


##### (10) Mobilogram peak picking using second derivative curvature
##### Edit the cell below to tune parameters for ATD peak picking


In [ ]:
# Which MS2 ATD to use for peak picking
# "ms2_same_mz": MS2 ATD extracted at the fragment m/z (same m/z as target)
# "ms2_by_precursor": MS2 ATD extracted using precursor_mz logic (only ideal for DDA, not HDMSE)
MS2_MODE = "ms2_same_mz"

# Savitzky-Golay smoothing parameters; used prior to taking derivatives (must be odd; will auto-fix if even)
SG_WINDOW = 7          # Recommended: 5-15
SG_POLY = 2            # Recommended: 2-3

# Minimum peak height threshold relative to the maximum signal in the trace
MIN_REL_HEIGHT = 0.01  # Recommended: 0.01-0.05

# Curvature prominence threshold (None disables)
CURV_PROM_REL = 0.01   # Try: 0.01-0.2

MAX_PEAKS = None       # Recommended to keep None for first-pass analyses

# Multi-Gaussian fitting parameters
CENTER_TOL_MS = 0.10
FWHM_PM_MS = 0.01
INCLUDE_BASELINE = True
MAXFEV = 20000

VERBOSE = True


##### Run the cell below to fit mobilograms and identify peaks


In [ ]:
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter, find_peaks, peak_widths
from scipy.optimize import curve_fit


FWHM_TO_SIGMA = 2.354820045

def _gauss(x, A, mu, sigma):
    return A * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

def gaussian_mixture(x, y0, *params):
    """params = [A1, mu1, sigma1, A2, mu2, sigma2, ...]"""
    y = np.full_like(x, y0, dtype=float)
    for i in range(0, len(params), 3):
        A, mu, sigma = params[i:i+3]
        y += _gauss(x, A, mu, sigma)
    return y

def second_derivative_peak_centers(dt, y, *,
                                   sg_window: int = 7,
                                   sg_poly: int = 2,
                                   min_rel_height: float = 0.01,
                                   curvature_prom_rel: float = 0.0,
                                   max_peaks: int | None = None):
    dt = np.asarray(dt, float)
    y = np.asarray(y, float)

    if len(dt) < 7 or np.nanmax(y) <= 0:
        return np.array([]), y, np.zeros_like(y), np.array([], dtype=int)

    if sg_window % 2 == 0:
        sg_window += 1
    sg_window = min(sg_window, len(y) if len(y) % 2 == 1 else len(y) - 1)
    sg_window = max(sg_window, 7)

    y_smooth = savgol_filter(y, window_length=sg_window, polyorder=sg_poly, mode="interp")
    delta = float(np.median(np.diff(dt))) if len(dt) > 3 else 0.01
    d2 = savgol_filter(y, window_length=sg_window, polyorder=sg_poly, deriv=2, delta=delta, mode="interp")
    curv = -d2

    curv_max = np.nanmax(curv) if np.isfinite(np.nanmax(curv)) else 0.0
    prom = curvature_prom_rel * curv_max if curvature_prom_rel > 0 else None

    cand_idx, _ = find_peaks(curv, prominence=prom)
    if cand_idx.size == 0:
        return np.array([]), y_smooth, d2, np.array([], dtype=int)

    y_max = np.nanmax(y_smooth)
    keep = [i for i in cand_idx if y_smooth[i] >= (min_rel_height * y_max)]
    peak_idx = np.array(keep, dtype=int)

    if peak_idx.size == 0:
        return np.array([]), y_smooth, d2, np.array([], dtype=int)

    if max_peaks is not None and peak_idx.size > max_peaks:
        order = np.argsort(y_smooth[peak_idx])[::-1]
        peak_idx = peak_idx[order[:max_peaks]]

    centers_dt = dt[peak_idx]
    centers_dt = np.array(sorted(centers_dt))
    peak_idx = np.array([np.argmin(np.abs(dt - c)) for c in centers_dt], dtype=int)
    return centers_dt, y_smooth, d2, peak_idx

def fit_atd_multi_gaussian(dt, y, centers_dt, *,
                           center_tol_ms: float = 0.1,
                           fwhm_center_peak: float | None = None,
                           fwhm_bounds_pm_ms: float = 0.01,
                           include_baseline: bool = True,
                           maxfev: int = 20000):
    dt = np.asarray(dt, float)
    y = np.clip(np.asarray(y, float), 0, None)

    if centers_dt is None or len(centers_dt) == 0 or np.nanmax(y) <= 0:
        return None

    if len(y) >= 7:
        wlen = 11 if len(y) >= 11 else (len(y) // 2) * 2 - 1
        wlen = max(wlen, 7)
        if wlen % 2 == 0:
            wlen -= 1
        y_s = savgol_filter(y, window_length=wlen, polyorder=2, mode="interp")
    else:
        y_s = y

    centers_dt = np.asarray(centers_dt, float)
    idx_centers = np.array([np.argmin(np.abs(dt - c)) for c in centers_dt], dtype=int)
    heights = y_s[idx_centers]
    h_idx = int(np.argmax(heights))
    mu_h = float(centers_dt[h_idx])

    if fwhm_center_peak is None:
        try:
            w = peak_widths(y_s, [idx_centers[h_idx]], rel_height=0.5)
            delta = float(np.median(np.diff(dt))) if len(dt) > 3 else 0.01
            fwhm_h = float(w[0][0] * delta)
            if not np.isfinite(fwhm_h) or fwhm_h <= 0:
                fwhm_h = 0.2
        except Exception:
            fwhm_h = 0.2
    else:
        fwhm_h = float(fwhm_center_peak)

    def fwhm_to_sigma(fwhm):
        return max(float(fwhm) / FWHM_TO_SIGMA, 1e-6)

    sigma_pm = float(fwhm_bounds_pm_ms) / FWHM_TO_SIGMA

    p0, lb, ub = [], [], []
    if include_baseline:
        y0_init = float(max(np.min(y_s), 0.0))
        p0.append(y0_init); lb.append(0.0); ub.append(float(np.max(y_s)))
    else:
        p0.append(0.0); lb.append(0.0); ub.append(0.0)

    for mu in centers_dt:
        mu = float(mu)
        A_init = float(y_s[np.argmin(np.abs(dt - mu))])
        A_init = max(A_init, 1.0)
        fwhm_n = fwhm_h * np.sqrt(max(mu, 1e-6) / max(mu_h, 1e-6))
        sigma_n = fwhm_to_sigma(fwhm_n)
        p0.extend([A_init, mu, sigma_n])
        lb.extend([0.0, mu - center_tol_ms, max(sigma_n - sigma_pm, 1e-6)])
        ub.extend([np.inf, mu + center_tol_ms, sigma_n + sigma_pm])

    try:
        popt, _ = curve_fit(lambda x, *pp: gaussian_mixture(x, *pp),
                            dt, y, p0=p0, bounds=(lb, ub), maxfev=maxfev)
    except Exception as e:
        return {"success": False, "error": str(e)}

    yfit = gaussian_mixture(dt, *popt)
    peaks = []
    params = popt[1:]
    for i in range(0, len(params), 3):
        A, mu, sigma = params[i:i+3]
        peaks.append({"A": float(A), "mu": float(mu), "sigma": float(sigma),
                      "FWHM": float(FWHM_TO_SIGMA * sigma)})

    ss_res = float(np.sum((y - yfit) ** 2))
    ss_tot_term = np.sum((y - np.mean(y)) ** 2)
    ss_tot = float(ss_tot_term) if ss_tot_term > 0 else np.nan
    r2 = 1.0 - ss_res / ss_tot if np.isfinite(ss_tot) else np.nan

    return {"success": True, "popt": popt, "yfit": yfit, "peaks": peaks, "r2": r2}

def run_peak_picking_all_atds(instances_df, atd_store, *,
                              ms2_mode: str = "ms2_same_mz",
                              sg_window: int = 7, sg_poly: int = 2,
                              min_rel_height: float = 0.01,
                              curvature_prom_rel: float = 0.0,
                              max_peaks=None,
                              center_tol_ms: float = 0.1,
                              fwhm_bounds_pm_ms: float = 0.01,
                              include_baseline: bool = True,
                              maxfev: int = 20000,
                              verbose: bool = True):
    rows = []
    fit_store = {}
    n_total = n_no_data = n_fit_fail = 0

    for (row_idx, lc_idx), d in atd_store.items():
        n_total += 1
        inst_match = instances_df[
            (instances_df["row_index"] == row_idx) &
            (instances_df["lc_peak_index"] == lc_idx)
        ]
        if inst_match.empty:
            feature_name = None; target_mz = np.nan
            rt_left = np.nan; rt_right = np.nan
        else:
            inst = inst_match.iloc[0]
            feature_name = inst.get("feature_name", None)
            target_mz = float(inst.get("target_mz", np.nan))
            rt_left = float(inst.get("rt_left", np.nan))
            rt_right = float(inst.get("rt_right", np.nan))

        ms1_dt = np.asarray(d.get("ms1_dt", []), float)
        ms1_y = np.asarray(d.get("ms1_atd", []), float)
        ms2_dt = np.asarray(d.get(f"{ms2_mode}_dt", []), float)
        ms2_y = np.asarray(d.get(f"{ms2_mode}_atd", []), float)

        if ms1_dt.size == 0 and ms2_dt.size == 0:
            n_no_data += 1
            continue

        if ms1_dt.size:
            ms1_centers, ms1_s, ms1_d2, ms1_idx = second_derivative_peak_centers(
                ms1_dt, ms1_y,
                sg_window=sg_window, sg_poly=sg_poly,
                min_rel_height=min_rel_height,
                curvature_prom_rel=curvature_prom_rel,
                max_peaks=max_peaks,
            )
            ms1_fit = fit_atd_multi_gaussian(
                ms1_dt, ms1_y, ms1_centers,
                center_tol_ms=center_tol_ms,
                fwhm_bounds_pm_ms=fwhm_bounds_pm_ms,
                include_baseline=include_baseline, maxfev=maxfev,
            )
        else:
            ms1_fit = None

        if ms2_dt.size:
            ms2_centers, ms2_s, ms2_d2, ms2_idx = second_derivative_peak_centers(
                ms2_dt, ms2_y,
                sg_window=sg_window, sg_poly=sg_poly,
                min_rel_height=min_rel_height,
                curvature_prom_rel=curvature_prom_rel,
                max_peaks=max_peaks,
            )
            ms2_fit = fit_atd_multi_gaussian(
                ms2_dt, ms2_y, ms2_centers,
                center_tol_ms=center_tol_ms,
                fwhm_bounds_pm_ms=fwhm_bounds_pm_ms,
                include_baseline=include_baseline, maxfev=maxfev,
            )
        else:
            ms2_fit = None

        if (ms1_fit is not None and not ms1_fit.get("success", True)) or \
           (ms2_fit is not None and not ms2_fit.get("success", True)):
            n_fit_fail += 1

        fit_store[(row_idx, lc_idx)] = {
            "feature_name": feature_name, "target_mz": target_mz,
            "rt_left": rt_left, "rt_right": rt_right, "ms2_mode": ms2_mode,
            "ms1": ms1_fit, "ms2": ms2_fit,
        }

        def _emit_peaks(level, fit):
            if fit is None:
                return
            if not fit.get("success", False):
                rows.append({
                    "row_index": row_idx, "lc_peak_index": lc_idx,
                    "feature_name": feature_name, "target_mz": target_mz,
                    "rt_left": rt_left, "rt_right": rt_right,
                    "level": level, "fit_success": False,
                    "fit_error": fit.get("error", None),
                    "r2": np.nan, "A": np.nan, "mu": np.nan,
                    "sigma": np.nan, "FWHM": np.nan, "n_peaks_fit": 0,
                })
                return
            peaks = fit.get("peaks", [])
            for p in peaks:
                rows.append({
                    "row_index": row_idx, "lc_peak_index": lc_idx,
                    "feature_name": feature_name, "target_mz": target_mz,
                    "rt_left": rt_left, "rt_right": rt_right,
                    "level": level, "fit_success": True, "fit_error": None,
                    "r2": fit.get("r2", np.nan),
                    "A": p.get("A", np.nan), "mu": p.get("mu", np.nan),
                    "sigma": p.get("sigma", np.nan), "FWHM": p.get("FWHM", np.nan),
                    "n_peaks_fit": len(peaks),
                })

        _emit_peaks("ms1", ms1_fit)
        _emit_peaks(ms2_mode, ms2_fit)

    peaks_df = pd.DataFrame(rows)
    if verbose:
        print(f"  ATDs processed: {n_total}")
        print(f"  ATDs with no data: {n_no_data}")
        print(f"  ATDs with any fit failure: {n_fit_fail}")
        print(f"  Output peaks rows: {len(peaks_df)}")
    return peaks_df, fit_store

peaks_df, atd_fit_store = run_peak_picking_all_atds(
    instances_df=instances_df,
    atd_store=atd_store,
    ms2_mode=MS2_MODE,
    sg_window=SG_WINDOW, sg_poly=SG_POLY,
    min_rel_height=MIN_REL_HEIGHT,
    curvature_prom_rel=CURV_PROM_REL,
    max_peaks=MAX_PEAKS,
    center_tol_ms=CENTER_TOL_MS,
    fwhm_bounds_pm_ms=FWHM_PM_MS,
    include_baseline=INCLUDE_BASELINE,
    maxfev=MAXFEV,
    verbose=VERBOSE,
)

display(peaks_df.head(20))

##### Visualize fitted and peak-picked ATDs for a selected MS1 feature
##### Edit the cell below to visualize the fitted ATDs


In [ ]:
# NOTE: Optimize peak fitting parameters for ATDs of interest before proceeding.
# The quality of the fitted peaks strongly influences downstream analysis.

FEATURE_NAME = "Plasma_144"
LC_PEAK_INDEX = None
ROW_INDEX = None

MS2_MODE = "ms2_same_mz"
USE_FIT_STORE_IF_AVAILABLE = True

SG_WINDOW = 7
SG_POLY = 2

PEAK_MARKER = "*"
PEAK_MARKER_SIZE = 70

RAW_LW = 3
SMOOTH_LW = 3
FIT_LW = 3

##### Run the cell below to generate fitted ATD plots


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
from scipy.signal import savgol_filter

rcParams["font.family"] = "Arial"


def _apply_axis_style(ax):
    ax.tick_params(axis="both", which="major", labelsize=10, width=2)
    for lab in ax.get_xticklabels() + ax.get_yticklabels():
        lab.set_fontweight("bold")
        lab.set_fontsize(12)
    for side in ("top", "right", "left", "bottom"):
        ax.spines[side].set_linewidth(3)


def _safe_savgol(y, window, poly):
    y = np.asarray(y, float)
    if y.size < 7:
        return y
    w = int(window)
    if w % 2 == 0:
        w += 1
    w = min(w, y.size if y.size % 2 == 1 else y.size - 1)
    w = max(w, 7)
    p = min(int(poly), w - 1)
    return savgol_filter(y, window_length=w, polyorder=p, mode="interp")


def plot_atd_raw_smooth_fit_peaks_for_feature(instances_df, atd_store, feature_name, *,
                                              ms2_mode="ms2_same_mz",
                                              lc_peak_index=None, row_index=None,
                                              use_fit_store_if_available=True,
                                              atd_fit_store=None,
                                              sg_window=7, sg_poly=2,
                                              peak_marker="*", peak_marker_size=90,
                                              raw_lw=1, smooth_lw=3, fit_lw=3):
    df = instances_df[instances_df["feature_name"].astype(str) == str(feature_name)].copy()
    if df.empty:
        raise ValueError(f"No instances for feature_name='{feature_name}'")
    if row_index is not None:
        df = df[df["row_index"] == row_index].copy()
        if df.empty:
            raise ValueError(f"No instances for row_index={row_index}")
    if lc_peak_index is not None:
        df = df[df["lc_peak_index"] == lc_peak_index].copy()
        if df.empty:
            raise ValueError(f"No instances for lc_peak_index={lc_peak_index}")

    df = df.sort_values(["row_index", "lc_peak_index"]).reset_index(drop=True)

    for _, inst in df.iterrows():
        key = (int(inst["row_index"]), int(inst["lc_peak_index"]))
        if key not in atd_store:
            print(f"Missing ATD data for key={key} (skipping)")
            continue
        d = atd_store[key]

        ms1_dt = np.asarray(d.get("ms1_dt", []), float)
        ms1_y = np.asarray(d.get("ms1_atd", []), float)
        ms2_dt = np.asarray(d.get(f"{ms2_mode}_dt", []), float)
        ms2_y = np.asarray(d.get(f"{ms2_mode}_atd", []), float)

        ms1_s = _safe_savgol(ms1_y, sg_window, sg_poly) if ms1_dt.size else np.array([])
        ms2_s = _safe_savgol(ms2_y, sg_window, sg_poly) if ms2_dt.size else np.array([])

        ms1_fit = ms2_fit = None
        if use_fit_store_if_available and atd_fit_store is not None and key in atd_fit_store:
            ms1_fit = atd_fit_store[key].get("ms1", None)
            ms2_fit = atd_fit_store[key].get("ms2", None)
        if ms1_fit is None and ms1_dt.size:
            centers, _, _, _ = second_derivative_peak_centers(ms1_dt, ms1_y, sg_window=sg_window, sg_poly=sg_poly)
            ms1_fit = fit_atd_multi_gaussian(ms1_dt, ms1_y, centers, include_baseline=True)
        if ms2_fit is None and ms2_dt.size:
            centers, _, _, _ = second_derivative_peak_centers(ms2_dt, ms2_y, sg_window=sg_window, sg_poly=sg_poly)
            ms2_fit = fit_atd_multi_gaussian(ms2_dt, ms2_y, centers, include_baseline=True)

        ms1_mu = np.array([p["mu"] for p in ms1_fit["peaks"]]) if (ms1_fit and ms1_fit.get("success")) else np.array([])
        ms2_mu = np.array([p["mu"] for p in ms2_fit["peaks"]]) if (ms2_fit and ms2_fit.get("success")) else np.array([])

        def _marker_y(dt, y_s, mu_list):
            ys = []
            for mu in mu_list:
                i = int(np.argmin(np.abs(dt - mu)))
                ys.append(float(y_s[i]) if y_s.size else np.nan)
            return np.array(ys, float)

        fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, figsize=(6.4, 6.9), sharex=False)
        fig.suptitle(
            f"{inst['feature_name']}\n"
            f"{inst['target_mz']:.4f} m/z\n"
            f"LC peak {int(inst['lc_peak_index'])} | RT Bounds: {inst['rt_left']:.2f} - {inst['rt_right']:.2f} min\n\n"
            f"Extracted Arrival Time Distributions",
            fontsize=14, fontweight="bold", y=0.98,
        )

        if ms1_dt.size:
            ax1.plot(ms1_dt, ms1_y, lw=raw_lw, color="black", label=" MS1 Raw Data")
            ax1.plot(ms1_dt, ms1_s, lw=smooth_lw, color="black", alpha=0.6, label="Smoothed Data")
            if ms1_fit and ms1_fit.get("success"):
                ax1.plot(ms1_dt, ms1_fit["yfit"], lw=fit_lw, color="tab:red", label="Gaussian Fit")
            if ms1_mu.size:
                ax1.scatter(ms1_mu, _marker_y(ms1_dt, ms1_s, ms1_mu),
                            marker=peak_marker, s=peak_marker_size,
                            color="purple", label="Identified Peaks", zorder=5)
        else:
            ax1.text(0.5, 0.5, "No MS1 ATD", ha="center", va="center", transform=ax1.transAxes)
        ax1.set_ylabel("Intensity", fontsize=12, fontweight="bold")
        _apply_axis_style(ax1)
        ax1.legend(loc="best", frameon=True, fontsize=12, edgecolor="black", facecolor="white")

        if ms2_dt.size:
            ax2.plot(ms2_dt, ms2_y, lw=raw_lw, color="tab:blue", label="MS2 Raw Data")
            ax2.plot(ms2_dt, ms2_s, lw=smooth_lw, color="tab:blue", alpha=0.6, label="Smoothed Data")
            if ms2_fit and ms2_fit.get("success"):
                ax2.plot(ms2_dt, ms2_fit["yfit"], lw=fit_lw, color="tab:red", label="Gaussian Fit")
            if ms2_mu.size:
                ax2.scatter(ms2_mu, _marker_y(ms2_dt, ms2_s, ms2_mu),
                            marker=peak_marker, s=peak_marker_size,
                            color="purple", label="Identified Peaks", zorder=5)
        else:
            ax2.text(0.5, 0.5, f"No MS2 ATD ({ms2_mode})", ha="center", va="center", transform=ax2.transAxes)
        ax2.set_xlabel("Drift Time (ms)", fontsize=12, fontweight="bold")
        ax2.set_ylabel("Intensity", fontsize=12, fontweight="bold")
        _apply_axis_style(ax2)
        ax2.legend(loc="best", frameon=True, fontsize=12, edgecolor="black", facecolor="white")

        for ax in (ax1, ax2):
            lines = ax.get_lines()
            if lines:
                y_max = max(float(np.max(ln.get_ydata())) for ln in lines if ln.get_ydata().size)
                ax.set_ylim(0, y_max * 1.2 if y_max > 0 else 1)

        plt.tight_layout(rect=[0, 0, 1, 0.94])
        plt.show()


plot_atd_raw_smooth_fit_peaks_for_feature(
    instances_df=instances_df, atd_store=atd_store,
    feature_name=FEATURE_NAME, ms2_mode=MS2_MODE,
    lc_peak_index=LC_PEAK_INDEX, row_index=ROW_INDEX,
    use_fit_store_if_available=USE_FIT_STORE_IF_AVAILABLE,
    atd_fit_store=(atd_fit_store if "atd_fit_store" in globals() else None),
    sg_window=SG_WINDOW, sg_poly=SG_POLY,
    peak_marker=PEAK_MARKER, peak_marker_size=PEAK_MARKER_SIZE,
    raw_lw=RAW_LW, smooth_lw=SMOOTH_LW, fit_lw=FIT_LW,
)


##### (11) Identification and scoring of candidate ISFs 

Running the cells below will compute the `isf_confidence` label for every detected feature.

**IM Metrics** 
- `drift_resolution = |Δμ| / 0.5*(FWHM_MS1 + FWHM_MS2)`: Separation betwteen ion populations in FWHM units. Rs ≥ 1.0 = baseline-resolved; 0.5–1.0 = partial
- `mobility_profile_pearson_r`: Pearson correlation coefficient between MS1 and offset-corrected MS2 mobilograms
- `ms1_unaligned_intensity_ratio`: MS1 mobilogram area in the unaligned drift window / area in the aligned window

**Spectral Similarity Metrics**
- `lc_coelution_pearson_r`: LC co-elution Pearson correlation coefficient r between precursor and fragment EICs; default threshold ≥ 0.8
- `ms2_subspectrum_match`: Fragment m/z present in precursor's MS2 within `MS2_MATCH_DA`
- `ms2_reverse_dot_product` / `ms2_matched_ratio`: Cosine similarity / matched fraction of the drift-time filtered fragment MS2 vs. precursor MS2. Thresholds `MS2_REVERSE_DP_MIN` / `MS2_MATCHED_RATIO_MIN`

**Confidence Levels:** 
- `high_im`: Rs ≥ 1 AND (mob_profile_pearson_r ≤ MOBILITY_CORR_MAX_FOR_ISF OR NaN) AND (int_ratio ≥ UNALIGNED_INTENSITY_MIN_RATIO OR NaN)
- `medium_im`: 0.5 ≤ Rs ≥ 1.0
- `high_spectral`: lc_coelution_pearson_r ≥ 0.8 AND ms2_subspectrum_match AND ms2_reverse_dot_product > 0.5 OR ms2_matched_ratio > 0.7
- `medium_spectral`: lc_coelution_pearson_r ≥ 0.8 AND ms2_subspectrum_match
- `low`: lc_coelution_pearson_r ≥ 0.8 
- `none`: No criteria met in either the IM or spectral similarity frameworks

##### Edit the cell below to tune IM scoring thresholds

In [ ]:
# Drift time offset in ms between MS1 and MS2 space
# MS2 ions travel faster than the corresponding MS1 ions under higher voltage
# Operationally the offset is voltage- and mass-dependent; a hard-set value is acceptable
DEIMOS_DT_OFFSET_MS = 0.1  # Recommended: 0.1-0.15

# Offset sign convention; True assumes MS2 ions always travel faster than MS1 under higher voltage
OFFSET_MAPS_MS2_TO_MS1 = True

# Drift time alignment tolerances
ABS_ALIGN_TOL_MS = 0.03  # Absolute floor (ms); recommended: 0.02-0.05
K_WIDTH = 0.30           # Scale factor on min(FWHM_MS1, FWHM_MS2); recommended: 0.25-0.50

# MS1 fragment-presence threshold
MIN_REL_TO_MAX_ATD = 0.01  # Recommended: 0.01 (i.e., 1% of max ATD intensity)
MIN_ABS_A = None           # Recommended: None

# IM confidence thresholds
RS_HIGH = 1.0                       # drift_resolution >= 1.0 -> high_im
RS_MEDIUM = 0.5                     # drift_resolution >= 0.5 -> medium_im
MOBILITY_CORR_MAX_FOR_ISF = 0.6     # mobility_profile_pearson_r must be <= this for IM-based confidence assignment
UNALIGNED_INTENSITY_MIN_RATIO = 0.3 # ms1_unaligned_intensity_ratio must be >= this for IM-based confidence assignment

# MS1 clustering
MS1_BIN_PPM = 10.0                # m/z clustering tolerance in ppm
TOP_K_CLUSTERS_PER_COMPONENT = 50 # None to keep all clusters
MIN_MS1_ROWS_PER_COMPONENT = 1    # Skip components with fewer MS1 rows than this
DT_HALF_WIDTH_FACTOR = 0.5        # Drift window half-width: mu +/- factor*FWHM

# OPTIONAL: only run alignment for a single feature name (None = align all)
FEATURE_NAME_FILTER = None

VERBOSE = True


##### Edit the cell below to tune spectral similarity scoring thresholds


In [ ]:
LC_COELUTION_MIN_R = 0.8       # LC-coelution Pearson r threshold
MS2_MATCH_DA = 0.02            # m/z match tolerance in Da
MS2_REVERSE_DP_MIN = 0.5       # Reverse-dot-product threshold
MS2_MATCHED_RATIO_MIN = 0.7    # Matched-ratio threshold

# LC-coelution smoothing 
COELUTION_SMOOTH_WINDOW = 5
COELUTION_SMOOTH_POLY = 2

# m/z bin width for projecting raw MS2 rows into a centroid-like spectrum
SPECTRAL_MZ_BIN_DA = 0.01

##### Run the cell below to compute ISF confidence labels for all features


In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

MS2_LEVEL = "ms2_same_mz"

def ms1_fragment_present_from_peaks(ms1_peaks_df, ms1_y, *,
                                    min_rel_to_max_atd: float = 0.01,
                                    min_abs_A: float | None = None) -> tuple[bool, str]:
    if ms1_peaks_df is None or ms1_peaks_df.empty:
        return False, "MS1 peaks missing"
    y_max = float(np.max(ms1_y)) if (ms1_y is not None and len(ms1_y) > 0) else 0.0
    if y_max <= 0:
        return False, "MS1 ATD max intensity <= 0"
    A_max = float(ms1_peaks_df["A"].max())
    if min_abs_A is not None and A_max < float(min_abs_A):
        return False, f"MS1 A_max={A_max:.3g} < min_abs_A={min_abs_A}"
    if A_max < float(min_rel_to_max_atd) * y_max:
        return False, f"MS1 A_max={A_max:.3g} < {min_rel_to_max_atd:.3g} * max(ATD)={y_max:.3g}"
    return True, "MS1 fragment present"

def label_ms2_alignment_widthaware_kdtree(ms1_peaks_df: pd.DataFrame,
                                          ms2_peaks_df: pd.DataFrame, *,
                                          dt_offset_ms: float,
                                          abs_align_tol_ms: float = 0.03,
                                          k_width: float = 0.30,
                                          offset_maps_ms2_to_ms1: bool = True,
                                          rs_medium: float = 0.5) -> pd.DataFrame:
    """cKDTree-backed nearest-neighbor lookup; returns Stack-A IM metrics."""
    cols = [
        "A", "mu", "sigma", "FWHM",
        "mu_ms2_to_ms1", "nearest_ms1_mu", "delta_to_nearest_ms1",
        "aligned_to_ms1_component", "alignment_label",
        "width_term_ms", "align_tol_ms_used",

        # IM metrics
        "drift_resolution", "drift_delta_pct", "r_im",
        "min_resolvable_delta_ms", "im_resolvable",
    ]
    if ms2_peaks_df is None or ms2_peaks_df.empty:
        return pd.DataFrame({c: [] for c in cols})

    ms2 = ms2_peaks_df[["A", "mu", "sigma", "FWHM"]].copy()
    shift = float(dt_offset_ms) if offset_maps_ms2_to_ms1 else -float(dt_offset_ms)
    ms2["mu_ms2_to_ms1"] = ms2["mu"].astype(float) + shift

    if ms1_peaks_df is None or ms1_peaks_df.empty:
        ms2["nearest_ms1_mu"] = np.nan
        ms2["delta_to_nearest_ms1"] = np.nan
        ms2["width_term_ms"] = np.nan
        ms2["align_tol_ms_used"] = float(abs_align_tol_ms)
        ms2["aligned_to_ms1_component"] = np.nan
        ms2["alignment_label"] = "unaligned"
        ms2["drift_resolution"] = np.nan
        ms2["drift_delta_pct"] = np.nan
        ms2["r_im"] = np.nan
        ms2["min_resolvable_delta_ms"] = np.nan
        ms2["im_resolvable"] = False
        return ms2[cols]

    ms1_mus = ms1_peaks_df["mu"].to_numpy(float)
    ms1_fwhm = ms1_peaks_df["FWHM"].to_numpy(float)
    ms2_fwhm = ms2["FWHM"].to_numpy(float)
    mapped = ms2["mu_ms2_to_ms1"].to_numpy(float)

    tree = cKDTree(ms1_mus[:, None])
    dist, idx = tree.query(mapped[:, None], k=1)
    nearest_mu = ms1_mus[idx]
    nearest_fwhm = ms1_fwhm[idx]

    finite = np.isfinite(nearest_fwhm) & np.isfinite(ms2_fwhm) & (nearest_fwhm > 0) & (ms2_fwhm > 0)
    width_term = np.where(finite, float(k_width) * np.minimum(nearest_fwhm, ms2_fwhm), 0.0)
    tol_used = np.maximum(float(abs_align_tol_ms), width_term)
    aligned_mask = dist <= tol_used

    ms2["nearest_ms1_mu"] = nearest_mu
    ms2["delta_to_nearest_ms1"] = dist
    ms2["width_term_ms"] = width_term
    ms2["align_tol_ms_used"] = tol_used
    ms2["aligned_to_ms1_component"] = np.where(aligned_mask, idx.astype(float), np.nan)
    ms2["alignment_label"] = np.where(aligned_mask, "aligned", "unaligned")

    # IM metrics
    fwhm_avg = np.where(finite, 0.5 * (nearest_fwhm + ms2_fwhm), np.nan)
    with np.errstate(invalid="ignore", divide="ignore"):
        rs = np.where(np.asarray(fwhm_avg) > 0, dist / np.asarray(fwhm_avg), np.nan)
        delta_pct = np.where(nearest_mu > 0, 100.0 * dist / nearest_mu, np.nan)
        r_im = np.where(nearest_fwhm > 0, nearest_mu / nearest_fwhm, np.nan)
        min_resolvable = np.where(np.asarray(r_im) > 0, nearest_mu / np.asarray(r_im), np.nan)
    ms2["drift_resolution"] = rs
    ms2["drift_delta_pct"] = delta_pct
    ms2["r_im"] = r_im
    ms2["min_resolvable_delta_ms"] = min_resolvable
    ms2["im_resolvable"] = np.where(np.isfinite(rs), rs >= float(rs_medium), False).astype(bool)
    return ms2[cols]


def mobility_profile_pearson(ms1_dt, ms1_y, ms2_dt, ms2_y, *,
                             dt_offset_ms: float,
                             offset_maps_ms2_to_ms1: bool = True,
                             drift_window_ms=None) -> float:
    """Pearson r between MS1 and offset-corrected MS2 mobilograms on a shared 1024-bin grid."""
    if ms1_dt is None or ms2_dt is None:
        return float("nan")
    ms1_dt = np.asarray(ms1_dt, float); ms1_y = np.asarray(ms1_y, float)
    ms2_dt = np.asarray(ms2_dt, float); ms2_y = np.asarray(ms2_y, float)
    if ms1_dt.size < 2 or ms2_dt.size < 2:
        return float("nan")
    shift = float(dt_offset_ms) if offset_maps_ms2_to_ms1 else -float(dt_offset_ms)
    ms2_dt_shift = ms2_dt + shift
    if drift_window_ms is not None:
        lo, hi = float(drift_window_ms[0]), float(drift_window_ms[1])
    else:
        lo = float(min(ms1_dt.min(), ms2_dt_shift.min()))
        hi = float(max(ms1_dt.max(), ms2_dt_shift.max()))
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return float("nan")
    grid = np.linspace(lo, hi, 1024)
    p1 = np.interp(grid, ms1_dt, ms1_y, left=0.0, right=0.0)
    p2 = np.interp(grid, ms2_dt_shift, ms2_y, left=0.0, right=0.0)
    if np.std(p1) <= 0 or np.std(p2) <= 0:
        return float("nan")
    r = float(np.corrcoef(p1, p2)[0, 1])
    return r if np.isfinite(r) else float("nan")

def unaligned_intensity_ratio(ms1_dt, ms1_y, *,
                              unaligned_mu: float, unaligned_fwhm: float,
                              aligned_mu: float, aligned_fwhm: float) -> float:
    """MS1 ATD area in the unaligned drift window / area in the aligned window."""
    if (ms1_dt is None or ms1_y is None or len(ms1_dt) == 0 or len(ms1_y) == 0
            or not np.isfinite(unaligned_mu) or not np.isfinite(unaligned_fwhm)
            or unaligned_fwhm <= 0):
        return float("nan")
    dt = np.asarray(ms1_dt, float); y = np.asarray(ms1_y, float)
    half_un = 0.5 * float(unaligned_fwhm)
    un_mask = (dt >= unaligned_mu - half_un) & (dt <= unaligned_mu + half_un)
    un_area = float(y[un_mask].sum())
    if not np.isfinite(aligned_mu) or not np.isfinite(aligned_fwhm) or aligned_fwhm <= 0:
        return float("nan")
    half_al = 0.5 * float(aligned_fwhm)
    al_mask = (dt >= aligned_mu - half_al) & (dt <= aligned_mu + half_al)
    al_area = float(y[al_mask].sum())
    if al_area <= 0:
        return float("nan")
    return un_area / al_area

def run_alignment_all_from_peaks_df(peaks_df, instances_df, atd_store, *,
                                    ms2_level: str = "ms2_same_mz",
                                    dt_offset_ms: float = 0.10,
                                    abs_align_tol_ms: float = 0.03,
                                    k_width: float = 0.30,
                                    offset_maps_ms2_to_ms1: bool = True,
                                    min_rel_to_max_atd: float = 0.01,
                                    min_abs_A=None,
                                    rs_medium: float = 0.5,
                                    feature_name_filter=None,
                                    verbose: bool = True) -> pd.DataFrame:
    if feature_name_filter is not None:
        peaks_df_use = peaks_df[peaks_df["feature_name"].astype(str) == str(feature_name_filter)].copy()
    else:
        peaks_df_use = peaks_df.copy()
    peaks_df_use = peaks_df_use[peaks_df_use["fit_success"] == True].copy()  # noqa: E712

    out_rows = []
    n_groups = n_skip_levels = n_skip_atd = n_skip_present = 0

    for (row_idx, lc_idx), g in peaks_df_use.groupby(["row_index", "lc_peak_index"], sort=False):
        n_groups += 1
        ms1_peaks = g[g["level"] == "ms1"][["A", "mu", "sigma", "FWHM"]].copy()
        ms2_peaks = g[g["level"] == ms2_level][["A", "mu", "sigma", "FWHM"]].copy()
        if ms1_peaks.empty or ms2_peaks.empty:
            n_skip_levels += 1
            continue

        key = (int(row_idx), int(lc_idx))
        if key not in atd_store:
            n_skip_atd += 1
            continue
        ms1_y = np.asarray(atd_store[key].get("ms1_atd", []), float)
        ms1_dt = np.asarray(atd_store[key].get("ms1_dt", []), float)
        ms2_y = np.asarray(atd_store[key].get(f"{ms2_level}_atd", []), float)
        ms2_dt = np.asarray(atd_store[key].get(f"{ms2_level}_dt", []), float)

        present, _ = ms1_fragment_present_from_peaks(
            ms1_peaks, ms1_y,
            min_rel_to_max_atd=min_rel_to_max_atd, min_abs_A=min_abs_A,
        )
        if not present:
            n_skip_present += 1
            continue

        labeled = label_ms2_alignment_widthaware_kdtree(
            ms1_peaks_df=ms1_peaks, ms2_peaks_df=ms2_peaks,
            dt_offset_ms=dt_offset_ms,
            abs_align_tol_ms=abs_align_tol_ms,
            k_width=k_width,
            offset_maps_ms2_to_ms1=offset_maps_ms2_to_ms1,
            rs_medium=rs_medium,
        )

        # Per-instance shared metrics
        mob_corr = mobility_profile_pearson(
            ms1_dt, ms1_y, ms2_dt, ms2_y,
            dt_offset_ms=dt_offset_ms,
            offset_maps_ms2_to_ms1=offset_maps_ms2_to_ms1,
        )
        aligned_fwhm_ms1 = float(ms1_peaks.iloc[0]["FWHM"]) if len(ms1_peaks) else float("nan")

        inst_match = instances_df[
            (instances_df["row_index"] == row_idx) &
            (instances_df["lc_peak_index"] == lc_idx)
        ]
        if not inst_match.empty:
            inst = inst_match.iloc[0]
            feature_name = inst.get("feature_name", None)
            target_mz = float(inst.get("target_mz", np.nan))
            rt_left = float(inst.get("rt_left", np.nan))
            rt_right = float(inst.get("rt_right", np.nan))
        else:
            feature_name = None; target_mz = np.nan
            rt_left = np.nan; rt_right = np.nan

        for _, r in labeled.iterrows():
            mu_ms2_to_ms1 = float(r.get("mu_ms2_to_ms1", np.nan))
            fwhm_ms2 = float(r.get("FWHM", np.nan))
            nearest_mu = float(r.get("nearest_ms1_mu", np.nan))
            ratio = unaligned_intensity_ratio(
                ms1_dt, ms1_y,
                unaligned_mu=mu_ms2_to_ms1, unaligned_fwhm=fwhm_ms2,
                aligned_mu=nearest_mu, aligned_fwhm=aligned_fwhm_ms1,
            )
            out_rows.append({
                "row_index": int(row_idx),
                "lc_peak_index": int(lc_idx),
                "feature_name": feature_name,
                "target_mz": target_mz,
                "rt_left": rt_left,
                "rt_right": rt_right,
                "A": float(r.get("A", np.nan)),
                "mu": float(r.get("mu", np.nan)),
                "sigma": float(r.get("sigma", np.nan)),
                "FWHM": fwhm_ms2,
                "mu_ms2_to_ms1": mu_ms2_to_ms1,
                "nearest_ms1_mu": nearest_mu,
                "delta_to_nearest_ms1": float(r.get("delta_to_nearest_ms1", np.nan)),
                "alignment_label": r.get("alignment_label"),
                # Stack-A IM extras
                "drift_resolution": float(r.get("drift_resolution", np.nan)),
                "drift_delta_pct": float(r.get("drift_delta_pct", np.nan)),
                "r_im": float(r.get("r_im", np.nan)),
                "min_resolvable_delta_ms": float(r.get("min_resolvable_delta_ms", np.nan)),
                "im_resolvable": bool(r.get("im_resolvable", False)),
                "mobility_profile_pearson_r": float(mob_corr),
                "ms1_unaligned_intensity_ratio": float(ratio),
            })

    _alignment_cols = [
        "row_index", "lc_peak_index", "feature_name", "target_mz",
        "rt_left", "rt_right", "A", "mu", "sigma", "FWHM",
        "mu_ms2_to_ms1", "nearest_ms1_mu", "delta_to_nearest_ms1",
        "alignment_label", "drift_resolution", "drift_delta_pct",
        "r_im", "min_resolvable_delta_ms", "im_resolvable",
        "mobility_profile_pearson_r", "ms1_unaligned_intensity_ratio",
    ]
    out_df = pd.DataFrame(out_rows) if out_rows else pd.DataFrame(columns=_alignment_cols)
    if verbose:
        print("ATD alignment complete.")
        print(f"  groups:                          {n_groups}")
        print(f"  skipped (missing ms1/ms2 level): {n_skip_levels}")
        print(f"  skipped (missing atd_store):     {n_skip_atd}")
        print(f"  skipped (MS1 not present):       {n_skip_present}")
        print(f"  output MS2 peak rows:            {len(out_df)}")
    return out_df

def _infer_file_key_for_instance(inst_row: pd.Series) -> str:
    candidates = []
    for col in ["file_stem", "file_key", "h5_stem", "stem", "file_name", "File Name", "h5_path", "path"]:
        if col in inst_row.index and pd.notna(inst_row[col]):
            candidates.append(str(inst_row[col]))
    for c in candidates:
        try:
            from pathlib import Path
            p = Path(c)
            key = p.stem if p.suffix.lower() == ".h5" else p.with_suffix("").name
        except Exception:
            key = c
        if key.lower().endswith(".h5"):
            key = key[:-3]
        if key in ms_data:
            return key
    raise KeyError(
        "Could not infer file key for instance. "
        "Ensure instances_df contains a file identifier (e.g., file_stem or file_name) matching ms_data keys."
    )

def _extract_ms1_ions_from_df(ms1_df, rt_min, rt_max, dt_min=None, dt_max=None, mz_min=None, mz_max=None):
    mask = (
        (ms1_df["retention_time"] >= rt_min) &
        (ms1_df["retention_time"] <= rt_max)
    )
    if dt_min is not None and dt_max is not None:
        mask &= (ms1_df["drift_time"] >= dt_min) & (ms1_df["drift_time"] <= dt_max)
    df = ms1_df[mask]
    if mz_min is not None:
        df = df[df["mz"] >= mz_min]
    if mz_max is not None:
        df = df[df["mz"] <= mz_max]
    return df

def extract_ms1_ions_for_unaligned(
        alignment_df, instances_df, ms_data, *,
        dt_half_width_factor: float = 0.5,
        mz_min=None, mz_max=None):
    """Extract MS1 ions at each unaligned MS2 component's drift-time window.
    Returns:
    unaligned_components_summary_df : pd.DataFrame
    ms1_ions_all_df : pd.DataFrame
    """
    ms1_ions_rows = []
    summary_rows = []
    unaligned = alignment_df[alignment_df["alignment_label"] == "unaligned"].copy()
    print(f"Unaligned MS2 ATD components: {len(unaligned)}")

    need = ["row_index", "lc_peak_index", "mu_ms2_to_ms1", "FWHM"]
    missing = [c for c in need if c not in unaligned.columns]
    if missing:
        raise ValueError(f"ALIGNMENT_DF missing required columns: {missing}")

    ms1_ions_rows = []
    summary_rows = []

    for comp_i, r in unaligned.reset_index(drop=True).iterrows():
        row_index = int(r["row_index"])
        lc_peak_index = int(r["lc_peak_index"])

        inst_match = instances_df[
            (instances_df["row_index"] == row_index) &
            (instances_df["lc_peak_index"] == lc_peak_index)
        ]
        if inst_match.empty:
            continue
        inst = inst_match.iloc[0]

        file_key = _infer_file_key_for_instance(inst)
        ms1_df = ms_data[file_key]["ms1"]

        if "rt_left" in inst.index and "rt_right" in inst.index:
            rt_min = float(inst["rt_left"]); rt_max = float(inst["rt_right"])
        elif "rt_left" in r.index and "rt_right" in r.index and pd.notna(r["rt_left"]) and pd.notna(r["rt_right"]):
            rt_min = float(r["rt_left"]); rt_max = float(r["rt_right"])
        else:
            raise KeyError("Could not find rt_left/rt_right in instances_df (or ALIGNMENT_DF fallback).")

        mu = float(r["mu_ms2_to_ms1"])
        fwhm = float(r["FWHM"]) if np.isfinite(r["FWHM"]) else np.nan
        if not np.isfinite(mu) or not np.isfinite(fwhm) or fwhm <= 0:
            continue

        _im_resolvable = bool(r.get("im_resolvable", False))
        dt_min = mu - dt_half_width_factor * fwhm
        dt_max = mu + dt_half_width_factor * fwhm

        # Skip IM scoring when MS2 mobility profiles are not resolved
        ms1_spec = _extract_ms1_ions_from_df(
            ms1_df, rt_min=rt_min, rt_max=rt_max,
            dt_min=dt_min if _im_resolvable else None,
            dt_max=dt_max if _im_resolvable else None,
            mz_min=mz_min, mz_max=mz_max,
        )

        summary_rows.append({
            "component_index": comp_i,
            "file_key": file_key,
            "row_index": row_index,
            "lc_peak_index": lc_peak_index,
            "feature_name": inst.get("feature_name", None),
            "target_mz": float(inst.get("target_mz", np.nan)),
            "rt_left": rt_min,
            "rt_right": rt_max,
            "mu_ms2_to_ms1": mu,
            "FWHM": fwhm,
            "dt_left": dt_min,
            "dt_right": dt_max,
            "n_ms1_rows": int(len(ms1_spec)),
            "n_ms1_unique_mz": int(ms1_spec["mz"].nunique()) if not ms1_spec.empty else 0,
            "im_resolvable": _im_resolvable,
        })

        if not ms1_spec.empty:
            tmp = ms1_spec.copy()
            tmp["component_index"] = comp_i
            tmp["file_key"] = file_key
            tmp["row_index"] = row_index
            tmp["lc_peak_index"] = lc_peak_index
            tmp["feature_name"] = inst.get("feature_name", None)
            tmp["target_mz"] = float(inst.get("target_mz", np.nan))
            tmp["rt_left"] = rt_min
            tmp["rt_right"] = rt_max
            tmp["dt_left"] = dt_min
            tmp["dt_right"] = dt_max
            tmp["mu_ms2_to_ms1"] = mu
            tmp["FWHM_ms2"] = fwhm
            ms1_ions_rows.append(tmp)

    _summary_cols = [
        "component_index", "file_key", "row_index", "lc_peak_index",
        "feature_name", "target_mz", "rt_left", "rt_right",
        "mu_ms2_to_ms1", "FWHM", "dt_left", "dt_right",
        "n_ms1_rows", "n_ms1_unique_mz", "im_resolvable",
    ]
    _summary_df = pd.DataFrame(summary_rows) if summary_rows else pd.DataFrame(columns=_summary_cols)
    _ions_df = pd.concat(ms1_ions_rows, ignore_index=True) if ms1_ions_rows else pd.DataFrame()
    return _summary_df, _ions_df

import numpy as np
import pandas as pd


def collapse_and_cluster_ms1_spectrum(ms1_spec_df: pd.DataFrame, *,
                                      ppm_tol: float = 10.0,
                                      intensity_col: str = "intensity",
                                      mz_col: str = "mz",
                                      top_k: int | None = None) -> pd.DataFrame:
    cols = ["cluster_id", "mz_center", "intensity_sum", "n_peaks", "mz_min", "mz_max"]
    if ms1_spec_df is None or ms1_spec_df.empty:
        return pd.DataFrame(columns=cols)

    collapsed = (
        ms1_spec_df[[mz_col, intensity_col]]
        .groupby(mz_col, as_index=False)[intensity_col]
        .sum()
        .sort_values(mz_col)
        .reset_index(drop=True)
        .rename(columns={mz_col: "mz", intensity_col: "intensity"})
    )
    mz = collapsed["mz"].to_numpy(float)
    I = collapsed["intensity"].to_numpy(float)

    clusters = []
    cluster_id = 0
    start_idx = 0
    mz_max = mz[0]

    def _ppm_window_local(mz_ref):
        delta = (ppm_tol * 1e-6) * mz_ref
        return mz_ref - delta, mz_ref + delta

    for i in range(1, len(mz)):
        _, max_allowed = _ppm_window_local(mz_max)
        if mz[i] <= max_allowed:
            mz_max = mz[i]
        else:
            idxs = np.arange(start_idx, i)
            mz_cluster = mz[idxs]; I_cluster = I[idxs]
            I_sum = float(I_cluster.sum())
            mz_center = float((mz_cluster * I_cluster).sum() / I_sum) if I_sum > 0 else float(mz_cluster.mean())
            clusters.append({
                "cluster_id": cluster_id, "mz_center": mz_center, "intensity_sum": I_sum,
                "n_peaks": int(len(idxs)), "mz_min": float(mz_cluster.min()), "mz_max": float(mz_cluster.max()),
            })
            cluster_id += 1
            start_idx = i
            mz_max = mz[i]

    idxs = np.arange(start_idx, len(mz))
    mz_cluster = mz[idxs]; I_cluster = I[idxs]
    I_sum = float(I_cluster.sum())
    mz_center = float((mz_cluster * I_cluster).sum() / I_sum) if I_sum > 0 else float(mz_cluster.mean())
    clusters.append({
        "cluster_id": cluster_id, "mz_center": mz_center, "intensity_sum": I_sum,
        "n_peaks": int(len(idxs)), "mz_min": float(mz_cluster.min()), "mz_max": float(mz_cluster.max()),
    })

    clustered = pd.DataFrame(clusters).sort_values("intensity_sum", ascending=False).reset_index(drop=True)
    if top_k is not None:
        clustered = clustered.head(int(top_k)).copy()
    return clustered


def cluster_ms1_spectra_per_unaligned_component(unaligned_components_summary_df, ms1_ions_all_df, *,
                                                ppm_tol: float = 10.0,
                                                top_k=None,
                                                min_rows: int = 1,
                                                verbose: bool = True) -> pd.DataFrame:
    if unaligned_components_summary_df is None or unaligned_components_summary_df.empty:
        return pd.DataFrame()
    if ms1_ions_all_df is None or ms1_ions_all_df.empty:
        return pd.DataFrame()

    out = []
    n_total = n_skipped = 0
    for _, meta in unaligned_components_summary_df.iterrows():
        comp = int(meta["component_index"])
        n_total += 1
        ms1_spec = ms1_ions_all_df[ms1_ions_all_df["component_index"] == comp].copy()
        if len(ms1_spec) < int(min_rows):
            n_skipped += 1; continue
        clustered = collapse_and_cluster_ms1_spectrum(
            ms1_spec_df=ms1_spec, ppm_tol=float(ppm_tol),
            intensity_col="intensity", mz_col="mz", top_k=top_k,
        )
        if clustered.empty:
            n_skipped += 1; continue
        clustered.insert(0, "component_index", comp)
        for col in ("file_key", "row_index", "lc_peak_index", "feature_name", "target_mz",
                    "rt_left", "rt_right", "dt_left", "dt_right", "mu_ms2_to_ms1"):
            clustered[col] = meta.get(col, np.nan)
        clustered["FWHM_ms2"] = meta.get("FWHM", np.nan)
        out.append(clustered)

    out_df = pd.concat(out, ignore_index=True) if out else pd.DataFrame()
    if verbose:
        print("MS1 m/z clustering complete.")
        print(f"  components:         {n_total}")
        print(f"  components skipped: {n_skipped}")
        print(f"  cluster rows:       {len(out_df)}")
        if len(out_df):
            print(f"  ppm_tol used:       {ppm_tol}")
    return out_df

import numpy as np
import pandas as pd
from scipy.signal import savgol_filter

def lc_coelution_pearson(ms1_df: pd.DataFrame, *,
                         precursor_mz: float, fragment_mz: float,
                         mz_ppm: float, rt_left: float, rt_right: float,
                         rt_bin_min: float = 0.01,
                         smooth_window: int = 5,
                         smooth_poly: int = 2) -> float:
    """Pearson r between smoothed precursor and fragment EICs over the LC peak window."""
    pre_rt, pre_y, _ = build_ms1_eic(ms1_df, precursor_mz, mz_ppm, rt_bin_min)
    frag_rt, frag_y, _ = build_ms1_eic(ms1_df, fragment_mz, mz_ppm, rt_bin_min)

    def _restrict(rt, y, lo, hi):
        if rt.size == 0:
            return rt, y
        m = (rt >= lo) & (rt <= hi)
        return rt[m], y[m]

    pre_rt, pre_y = _restrict(pre_rt, pre_y, rt_left, rt_right)
    frag_rt, frag_y = _restrict(frag_rt, frag_y, rt_left, rt_right)
    if pre_rt.size < 2 or frag_rt.size < 2:
        return float("nan")

    rt_lo = float(min(pre_rt.min(), frag_rt.min()))
    rt_hi = float(max(pre_rt.max(), frag_rt.max()))
    if not np.isfinite(rt_lo) or not np.isfinite(rt_hi) or rt_hi <= rt_lo:
        return float("nan")
    n = max(int(np.ceil((rt_hi - rt_lo) / float(rt_bin_min))) + 1, 5)
    grid = np.linspace(rt_lo, rt_hi, n)

    pre = np.interp(grid, pre_rt, pre_y, left=0.0, right=0.0)
    frag = np.interp(grid, frag_rt, frag_y, left=0.0, right=0.0)

    w = max(3, int(smooth_window))
    if w % 2 == 0:
        w += 1
    if w > pre.size:
        w = pre.size if pre.size % 2 == 1 else max(3, pre.size - 1)
    p = max(1, min(int(smooth_poly), w - 1))
    if w >= 5 and pre.size >= w:
        pre = savgol_filter(pre, window_length=w, polyorder=p)
        frag = savgol_filter(frag, window_length=w, polyorder=p)
    if np.std(pre) <= 0 or np.std(frag) <= 0:
        return float("nan")
    r = float(np.corrcoef(pre, frag)[0, 1])
    return r if np.isfinite(r) else float("nan")

def _bin_spectrum(df: pd.DataFrame, *, mz_bin_da: float) -> pd.DataFrame:
    """Project a raw MS row set into a centroid-like (mz_bin_center, intensity_sum) spectrum."""
    if df is None or df.empty:
        return pd.DataFrame(columns=["mz", "intensity"])
    mz = df["mz"].to_numpy(float)
    inten = df["intensity"].to_numpy(float)
    if mz.size == 0:
        return pd.DataFrame(columns=["mz", "intensity"])
    bin_da = float(mz_bin_da)
    bins = np.round(mz / bin_da).astype(int)
    keys, inv = np.unique(bins, return_inverse=True)
    sums = np.bincount(inv, weights=inten)
    out = pd.DataFrame({"mz": keys * bin_da, "intensity": sums})
    return out[out["intensity"] > 0].sort_values("intensity", ascending=False).reset_index(drop=True)

def extract_ms2_subspectrum(ms2_df: pd.DataFrame, *,
                            rt_left: float, rt_right: float,
                            dt_left: float | None = None,
                            dt_right: float | None = None,
                            mz_bin_da: float = 0.01) -> pd.DataFrame:
    """Bin MS2 rows inside the (rt, drift) box into a centroid-like (mz, intensity) spectrum."""
    if ms2_df is None or ms2_df.empty:
        return pd.DataFrame(columns=["mz", "intensity"])
    sub = ms2_df[
        (ms2_df["retention_time"] >= float(rt_left)) &
        (ms2_df["retention_time"] <= float(rt_right))
    ]
    if dt_left is not None and dt_right is not None:
        sub = sub[(sub["drift_time"] >= float(dt_left)) & (sub["drift_time"] <= float(dt_right))]
    return _bin_spectrum(sub, mz_bin_da=mz_bin_da)


def _coerce_spectrum(spec: pd.DataFrame | None) -> tuple[np.ndarray, np.ndarray]:
    if spec is None or len(spec) == 0:
        return np.empty(0, float), np.empty(0, float)
    mz = spec["mz"].to_numpy(float); inten = spec["intensity"].to_numpy(float)
    finite = np.isfinite(mz) & np.isfinite(inten) & (inten > 0)
    return mz[finite], inten[finite]

def _l2(x: np.ndarray) -> np.ndarray:
    n = float(np.linalg.norm(x))
    if n <= 0:
        return np.zeros_like(x)
    return x / n

def _match_indices(query_mz: np.ndarray, library_mz: np.ndarray, *, mz_tol_da: float) -> np.ndarray:
    if query_mz.size == 0 or library_mz.size == 0:
        return np.full(query_mz.shape, -1, dtype=np.intp)
    order = np.argsort(library_mz, kind="mergesort")
    sorted_lib = library_mz[order]
    pos = np.searchsorted(sorted_lib, query_mz)
    out = np.full(query_mz.shape, -1, dtype=np.intp)
    n = sorted_lib.size
    for i in range(query_mz.size):
        candidates = []
        if 0 <= pos[i] < n:
            candidates.append((abs(sorted_lib[pos[i]] - query_mz[i]), int(order[pos[i]])))
        if pos[i] - 1 >= 0:
            candidates.append((abs(sorted_lib[pos[i] - 1] - query_mz[i]), int(order[pos[i] - 1])))
        if not candidates:
            continue
        d, idx = min(candidates, key=lambda t: t[0])
        if d <= mz_tol_da:
            out[i] = idx
    return out

def reverse_dot_product(query_spec: pd.DataFrame | None,
                        library_spec: pd.DataFrame | None, *,
                        mz_tol_da: float = 0.02) -> float:
    qmz, qi = _coerce_spectrum(query_spec)
    lmz, li = _coerce_spectrum(library_spec)
    if qmz.size == 0 or lmz.size == 0:
        return 0.0
    matches = _match_indices(qmz, lmz, mz_tol_da=mz_tol_da)
    matched = matches >= 0
    if not matched.any():
        return 0.0
    q_vec = _l2(qi[matched])
    l_vec = _l2(li[matches[matched]])
    return float(np.dot(q_vec, l_vec))

def matched_ratio(query_spec: pd.DataFrame | None,
                  library_spec: pd.DataFrame | None, *,
                  mz_tol_da: float = 0.02) -> float:
    qmz, _ = _coerce_spectrum(query_spec)
    lmz, _ = _coerce_spectrum(library_spec)
    if qmz.size == 0 or lmz.size == 0:
        return 0.0
    matches = _match_indices(qmz, lmz, mz_tol_da=mz_tol_da)
    return float((matches >= 0).sum() / qmz.size)

def ms2_subspectrum_contains(precursor_ms2: pd.DataFrame | None,
                             fragment_mz: float, *,
                             mz_tol_da: float = 0.02) -> bool:
    pmz, _ = _coerce_spectrum(precursor_ms2)
    if pmz.size == 0 or not np.isfinite(fragment_mz):
        return False
    return bool(np.min(np.abs(pmz - float(fragment_mz))) <= float(mz_tol_da))

def score_stack_b_per_component(unaligned_components_summary_df: pd.DataFrame,
                                clustered_ms1_all_df: pd.DataFrame,
                                ms_data: dict, *,
                                mz_ppm: float,
                                rt_bin_min: float,
                                dt_offset_ms: float,
                                offset_maps_ms2_to_ms1: bool,
                                ms2_match_da: float,
                                coelution_smooth_window: int,
                                coelution_smooth_poly: int,
                                mz_bin_da: float) -> pd.DataFrame:
    """Compute Stack-B metrics for each unaligned component."""
    if unaligned_components_summary_df is None or unaligned_components_summary_df.empty:
        return pd.DataFrame()

    components = unaligned_components_summary_df.copy()
    n = len(components)
    lc_pearson = np.full(n, np.nan, float)
    subspec_match = np.zeros(n, bool)
    rev_dp = np.full(n, np.nan, float)
    matched_r = np.full(n, np.nan, float)
    precursor_mz_used = np.full(n, np.nan, float)

    cluster_lookup: dict[int, pd.DataFrame] = {}
    if clustered_ms1_all_df is not None and not clustered_ms1_all_df.empty \
            and "component_index" in clustered_ms1_all_df.columns:
        for cidx, grp in clustered_ms1_all_df.groupby("component_index", sort=False):
            cluster_lookup[int(cidx)] = grp.sort_values("intensity_sum", ascending=False, kind="mergesort")

    for i, comp in enumerate(components.itertuples(index=False)):
        comp_idx = int(getattr(comp, "component_index"))
        target_mz = float(getattr(comp, "target_mz"))
        rt_left = float(getattr(comp, "rt_left"))
        rt_right = float(getattr(comp, "rt_right"))
        mu_ms1 = float(getattr(comp, "mu_ms2_to_ms1"))
        fwhm = float(getattr(comp, "FWHM"))
        file_key = str(getattr(comp, "file_key"))

        offset = float(dt_offset_ms) if offset_maps_ms2_to_ms1 else -float(dt_offset_ms)
        ms2_mu = mu_ms1 - offset
        _im_resolvable = bool(getattr(comp, "im_resolvable", False))
        if np.isfinite(mu_ms1) and np.isfinite(fwhm) and _im_resolvable:
            dt_lo = ms2_mu - 0.5 * fwhm
            dt_hi = ms2_mu + 0.5 * fwhm
        else:
            dt_lo = dt_hi = None

        cluster_grp = cluster_lookup.get(comp_idx)
        if cluster_grp is None or cluster_grp.empty:
            continue
        precursor_mz = float(cluster_grp.iloc[0]["mz_center"])
        precursor_mz_used[i] = precursor_mz

        ms1_df = ms_data[file_key]["ms1"]
        ms2_df = ms_data[file_key]["ms2"]

        try:
            lc_pearson[i] = lc_coelution_pearson(
                ms1_df, precursor_mz=precursor_mz, fragment_mz=target_mz,
                mz_ppm=mz_ppm, rt_left=rt_left, rt_right=rt_right,
                rt_bin_min=rt_bin_min,
                smooth_window=coelution_smooth_window,
                smooth_poly=coelution_smooth_poly,
            )
        except Exception:
            lc_pearson[i] = float("nan")

        try:
            precursor_ms2 = extract_ms2_subspectrum(
                ms2_df, rt_left=rt_left, rt_right=rt_right,
                mz_bin_da=mz_bin_da,
            )
        except Exception:
            precursor_ms2 = pd.DataFrame()
        subspec_match[i] = ms2_subspectrum_contains(precursor_ms2, target_mz, mz_tol_da=ms2_match_da)

        try:
            fragment_ms2 = extract_ms2_subspectrum(
                ms2_df, rt_left=rt_left, rt_right=rt_right,
                dt_left=dt_lo, dt_right=dt_hi,
                mz_bin_da=mz_bin_da,
            )
        except Exception:
            fragment_ms2 = pd.DataFrame()

        rev_dp[i] = reverse_dot_product(fragment_ms2, precursor_ms2, mz_tol_da=ms2_match_da)
        matched_r[i] = matched_ratio(fragment_ms2, precursor_ms2, mz_tol_da=ms2_match_da)

    components["precursor_mz_top_candidate"] = precursor_mz_used
    components["lc_coelution_pearson_r"] = lc_pearson
    components["ms2_subspectrum_match"] = subspec_match
    components["ms2_reverse_dot_product"] = rev_dp
    components["ms2_matched_ratio"] = matched_r
    return components

import numpy as np
import pandas as pd

def merge_stack_a_into_components(components: pd.DataFrame, alignment_df: pd.DataFrame) -> pd.DataFrame:
    """Join mobility-profile Pearson, unaligned-intensity ratio, and drift_resolution
    onto the components frame using the (row_index, lc_peak_index, mu_ms2_to_ms1) key."""
    if alignment_df is None or alignment_df.empty:
        for col, default in (
            ("mobility_profile_pearson_r", float("nan")),
            ("ms1_unaligned_intensity_ratio", float("nan")),
            ("drift_resolution", float("nan")),
            ("drift_delta_pct", float("nan")),
            ("im_resolvable", False),
        ):
            components[col] = default
        return components
    keep = (
        "row_index", "lc_peak_index", "mu_ms2_to_ms1",
        "drift_resolution", "drift_delta_pct", "im_resolvable",
        "mobility_profile_pearson_r", "ms1_unaligned_intensity_ratio",
    )
    sub = alignment_df[[c for c in keep if c in alignment_df.columns]].copy()
    sub["_mu_key"] = sub["mu_ms2_to_ms1"].round(6)
    components = components.copy()
    components["_mu_key"] = components["mu_ms2_to_ms1"].round(6)
    merged = components.merge(
        sub.drop(columns=["mu_ms2_to_ms1"]),
        on=["row_index", "lc_peak_index", "_mu_key"],
        how="left", suffixes=("", "_align"),
    ).drop(columns=["_mu_key"])
    return merged

def compose_isf_confidence(row: pd.Series, *,
                           rs_high: float, rs_medium: float,
                           mobility_corr_max_for_isf: float,
                           unaligned_intensity_min_ratio: float,
                           lc_coelution_min_r: float,
                           ms2_reverse_dp_min: float,
                           ms2_matched_ratio_min: float) -> str:
    rs = float(row.get("drift_resolution", float("nan")))
    mob_r = float(row.get("mobility_profile_pearson_r", float("nan")))
    int_ratio = float(row.get("ms1_unaligned_intensity_ratio", float("nan")))

    lc_r = float(row.get("lc_coelution_pearson_r", float("nan")))
    subspec = bool(row.get("ms2_subspectrum_match", False))
    rdp = float(row.get("ms2_reverse_dot_product", float("nan")))
    mr = float(row.get("ms2_matched_ratio", float("nan")))

    stack_a_supplementary = (
        (not np.isfinite(mob_r) or mob_r <= float(mobility_corr_max_for_isf))
        and (not np.isfinite(int_ratio) or int_ratio >= float(unaligned_intensity_min_ratio))
    )
    if np.isfinite(rs) and rs >= float(rs_high) and stack_a_supplementary:
        return "high_im"
    if np.isfinite(rs) and rs >= float(rs_medium) and stack_a_supplementary:
        return "medium_im"

    coelutes = np.isfinite(lc_r) and lc_r >= float(lc_coelution_min_r)
    sub_passes = subspec
    level_1 = (np.isfinite(rdp) and rdp > float(ms2_reverse_dp_min)) or \
              (np.isfinite(mr) and mr > float(ms2_matched_ratio_min))
    if coelutes and sub_passes and level_1:
        return "high_spectral"
    if coelutes and sub_passes:
        return "medium_spectral"
    if coelutes:
        return "low"
    return "none"

# Compose the final isf_confidence label
def find_unscored_instances(instances_df, alignment_df):
    # Return instances whose (row_index, lc_peak_index) have no rows in alignment_df.
    if instances_df is None or instances_df.empty:
        return pd.DataFrame()
    if alignment_df is None or alignment_df.empty or 'row_index' not in alignment_df.columns:
        return instances_df.copy().reset_index(drop=True)
    scored_keys = set(
        zip(alignment_df['row_index'].astype(int),
            alignment_df['lc_peak_index'].astype(int))
    )
    mask = [
        (int(r['row_index']), int(r['lc_peak_index'])) not in scored_keys
        for _, r in instances_df.iterrows()
    ]
    return instances_df[mask].copy().reset_index(drop=True)

def build_stub_components(unscored_df, ms_data, *, offset=0):

    # Build unaligned_components_summary_df for instances with no MS1 ATD peaks
    rows = []
    for i, inst in enumerate(unscored_df.itertuples(index=False)):
        try:
            file_key = _infer_file_key_for_instance(pd.Series(inst._asdict()))
        except Exception:
            continue
        rows.append({
            'component_index': offset + i,
            'file_key': file_key,
            'row_index': int(getattr(inst, 'row_index')),
            'lc_peak_index': int(getattr(inst, 'lc_peak_index')),
            'feature_name': getattr(inst, 'feature_name', None),
            'target_mz': float(getattr(inst, 'target_mz', float('nan'))),
            'rt_left': float(getattr(inst, 'rt_left', float('nan'))),
            'rt_right': float(getattr(inst, 'rt_right', float('nan'))),
            'mu_ms2_to_ms1': float('nan'),
            'FWHM': float('nan'),
            'dt_left': float('nan'),
            'dt_right': float('nan'),
            'n_ms1_rows': 0,
            'n_ms1_unique_mz': 0,
            'im_resolvable': False,
        })
    return pd.DataFrame(rows)


def extract_ms1_ions_rt_only(stub_components_df, ms_data, *,
                              mz_min=None, mz_max=None, verbose=True):
    
    # Extract MS1 ions over the LC peak RT window 
    if stub_components_df is None or stub_components_df.empty:
        return pd.DataFrame()
    ms1_ions_rows = []
    for _, comp in stub_components_df.iterrows():
        comp_idx = int(comp['component_index'])
        file_key = str(comp['file_key'])
        rt_left = float(comp['rt_left'])
        rt_right = float(comp['rt_right'])
        if not np.isfinite(rt_left) or not np.isfinite(rt_right):
            continue
        if file_key not in ms_data:
            continue
        ms1_df = ms_data[file_key]['ms1']
        sub = ms1_df[
            (ms1_df['retention_time'] >= rt_left) &
            (ms1_df['retention_time'] <= rt_right)
        ].copy()
        if mz_min is not None:
            sub = sub[sub['mz'] >= float(mz_min)]
        if mz_max is not None:
            sub = sub[sub['mz'] <= float(mz_max)]
        if sub.empty:
            continue
        sub['component_index'] = comp_idx
        sub['file_key'] = file_key
        sub['row_index'] = int(comp['row_index'])
        sub['lc_peak_index'] = int(comp['lc_peak_index'])
        sub['feature_name'] = comp.get('feature_name', None)
        sub['target_mz'] = float(comp.get('target_mz', float('nan')))
        sub['rt_left'] = rt_left
        sub['rt_right'] = rt_right
        sub['dt_left'] = float('nan')
        sub['dt_right'] = float('nan')
        sub['mu_ms2_to_ms1'] = float('nan')
        sub['FWHM_ms2'] = float('nan')
        ms1_ions_rows.append(sub)
    result = pd.concat(ms1_ions_rows, ignore_index=True) if ms1_ions_rows else pd.DataFrame()
    if verbose:
        n = stub_components_df['component_index'].nunique() if len(stub_components_df) else 0
        print(f'RT-only MS1 extraction: {n} stub component(s), {len(result)} MS1 rows')
    return result

MS2_LEVEL = "ms2_same_mz"

# Aign MS1/MS2 and compute IM metrics
ms2_alignment_all_df = run_alignment_all_from_peaks_df(
    peaks_df=peaks_df, instances_df=instances_df, atd_store=atd_store,
    ms2_level=MS2_LEVEL,
    dt_offset_ms=DEIMOS_DT_OFFSET_MS,
    abs_align_tol_ms=ABS_ALIGN_TOL_MS,
    k_width=K_WIDTH,
    offset_maps_ms2_to_ms1=OFFSET_MAPS_MS2_TO_MS1,
    min_rel_to_max_atd=MIN_REL_TO_MAX_ATD,
    min_abs_A=MIN_ABS_A,
    rs_medium=RS_MEDIUM,
    feature_name_filter=FEATURE_NAME_FILTER,
    verbose=VERBOSE,
)

# Extract MS1 ions at each unaligned fragment drift window
unaligned_components_summary_df, ms1_ions_all_df = extract_ms1_ions_for_unaligned(
    alignment_df=ms2_alignment_all_df,
    instances_df=instances_df,
    ms_data=ms_data,
    dt_half_width_factor=DT_HALF_WIDTH_FACTOR,
)

# Bin MS1 ions into m/z clusters 
clustered_ms1_all_df = cluster_ms1_spectra_per_unaligned_component(
    unaligned_components_summary_df=unaligned_components_summary_df,
    ms1_ions_all_df=ms1_ions_all_df,
    ppm_tol=MS1_BIN_PPM,
    top_k=TOP_K_CLUSTERS_PER_COMPONENT,
    min_rows=MIN_MS1_ROWS_PER_COMPONENT,
    verbose=VERBOSE,
)

# Spectral-similarity scoring
components_with_stack_b_df = score_stack_b_per_component(
    unaligned_components_summary_df=unaligned_components_summary_df,
    clustered_ms1_all_df=clustered_ms1_all_df,
    ms_data=ms_data,
    mz_ppm=MZ_PPM,
    rt_bin_min=RT_BIN_MIN,
    dt_offset_ms=DEIMOS_DT_OFFSET_MS,
    offset_maps_ms2_to_ms1=OFFSET_MAPS_MS2_TO_MS1,
    ms2_match_da=MS2_MATCH_DA,
    coelution_smooth_window=COELUTION_SMOOTH_WINDOW,
    coelution_smooth_poly=COELUTION_SMOOTH_POLY,
    mz_bin_da=SPECTRAL_MZ_BIN_DA,
)

# Merge Stack A metrics and assign isf_confidence label
components_scored = merge_stack_a_into_components(
    components_with_stack_b_df, ms2_alignment_all_df
)
components_scored["isf_confidence"] = [
    compose_isf_confidence(
        row,
        rs_high=RS_HIGH,
        rs_medium=RS_MEDIUM,
        mobility_corr_max_for_isf=MOBILITY_CORR_MAX_FOR_ISF,
        unaligned_intensity_min_ratio=UNALIGNED_INTENSITY_MIN_RATIO,
        lc_coelution_min_r=LC_COELUTION_MIN_R,
        ms2_reverse_dp_min=MS2_REVERSE_DP_MIN,
        ms2_matched_ratio_min=MS2_MATCHED_RATIO_MIN,
    )
    for _, row in components_scored.iterrows()
]

show_cols = [
    "component_index", "feature_name", "target_mz",
    "drift_resolution", "mobility_profile_pearson_r", "ms1_unaligned_intensity_ratio",
    "lc_coelution_pearson_r", "ms2_subspectrum_match",
    "ms2_reverse_dot_product", "ms2_matched_ratio",
    "isf_confidence",
]
show_cols = [c for c in show_cols if c in components_scored.columns]

# Spectral similarity fallback
_unscored = find_unscored_instances(instances_df, ms2_alignment_all_df)
if not _unscored.empty:
    if VERBOSE:
        print(
            f"\nStack-B fallback: {len(_unscored)} instance(s) had no MS1 ATD peaks "
            "and will be scored via LC co-elution + MS2 similarity only."
        )
    _stub_offset = (
        int(unaligned_components_summary_df['component_index'].max()) + 1
        if not unaligned_components_summary_df.empty else 0
    )
    _stub_comps = build_stub_components(_unscored, ms_data, offset=_stub_offset)
    _stub_ms1_ions = extract_ms1_ions_rt_only(_stub_comps, ms_data, verbose=VERBOSE)
    _stub_clustered = cluster_ms1_spectra_per_unaligned_component(
        _stub_comps, _stub_ms1_ions,
        ppm_tol=MS1_BIN_PPM, top_k=TOP_K_CLUSTERS_PER_COMPONENT,
        min_rows=MIN_MS1_ROWS_PER_COMPONENT, verbose=VERBOSE,
    )
    _stub_stack_b = score_stack_b_per_component(
        _stub_comps, _stub_clustered, ms_data,
        mz_ppm=MZ_PPM, rt_bin_min=RT_BIN_MIN,
        dt_offset_ms=DEIMOS_DT_OFFSET_MS, offset_maps_ms2_to_ms1=OFFSET_MAPS_MS2_TO_MS1,
        ms2_match_da=MS2_MATCH_DA, coelution_smooth_window=COELUTION_SMOOTH_WINDOW,
        coelution_smooth_poly=COELUTION_SMOOTH_POLY, mz_bin_da=SPECTRAL_MZ_BIN_DA,
    )
    for _col in ('drift_resolution', 'drift_delta_pct',
                 'mobility_profile_pearson_r', 'ms1_unaligned_intensity_ratio'):
        _stub_stack_b[_col] = float('nan')
    _stub_stack_b['im_resolvable'] = False
    _stub_stack_b['isf_confidence'] = [
        compose_isf_confidence(
            row, rs_high=RS_HIGH, rs_medium=RS_MEDIUM,
            mobility_corr_max_for_isf=MOBILITY_CORR_MAX_FOR_ISF,
            unaligned_intensity_min_ratio=UNALIGNED_INTENSITY_MIN_RATIO,
            lc_coelution_min_r=LC_COELUTION_MIN_R,
            ms2_reverse_dp_min=MS2_REVERSE_DP_MIN,
            ms2_matched_ratio_min=MS2_MATCHED_RATIO_MIN,
        )
        for _, row in _stub_stack_b.iterrows()
    ]
    unaligned_components_summary_df = pd.concat(
        [unaligned_components_summary_df, _stub_comps], ignore_index=True
    )
    if not _stub_ms1_ions.empty:
        ms1_ions_all_df = pd.concat([ms1_ions_all_df, _stub_ms1_ions], ignore_index=True)
    components_scored = pd.concat([components_scored, _stub_stack_b], ignore_index=True)
    show_cols = [c for c in show_cols if c in components_scored.columns]

print("ISF-confidence distribution:")
print(components_scored["isf_confidence"].value_counts(dropna=False).to_string())
display(components_scored[show_cols].head(25))

##### (12) Visualize the set of candidate MS1 ions associated with a selected ISF candidate


In [ ]:
COMPONENT_INDEX = 1 # Edit to visualize different candidate ion
TOP_N_IONS = 10
MZ_MIN_PLOT = None
MZ_MAX_PLOT = None

##### Run the cell below to generate MS1 spectrum plot


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams

rcParams["font.family"] = "Arial"

def _apply_axis_style(ax):
    ax.tick_params(axis="both", which="major", labelsize=10, width=1.5)
    for lab in ax.get_xticklabels() + ax.get_yticklabels():
        lab.set_fontweight("bold"); lab.set_fontsize(10)
    for side in ("top", "right", "left", "bottom"):
        ax.spines[side].set_linewidth(3)

def get_component_ms1_spectrum_df(ms1_ions_all_df, component_index, mz_min=None, mz_max=None):
    df = ms1_ions_all_df[ms1_ions_all_df["component_index"] == component_index].copy()
    if df.empty:
        raise ValueError(f"No MS1 ions found for component_index={component_index}")
    if mz_min is not None:
        df = df[df["mz"] >= float(mz_min)]
    if mz_max is not None:
        df = df[df["mz"] <= float(mz_max)]
    return (df.groupby("mz", as_index=False)["intensity"]
              .sum()
              .sort_values("intensity", ascending=False))

def plot_ms1_spectrum_atd_style(spec_df, title, top_n=50):
    if spec_df.empty:
        print("Empty spectrum after filtering.")
        return
    plot_df = spec_df.head(int(top_n)).sort_values("mz").copy()
    fig, ax = plt.subplots(figsize=(6.4, 4))
    ax.vlines(plot_df["mz"].to_numpy(float), 0, plot_df["intensity"].to_numpy(float), linewidth=2)
    ax.set_xlabel("m/z", fontsize=12, fontweight="bold")
    ax.set_ylabel("Intensity", fontsize=12, fontweight="bold")
    ax.set_title(title, fontsize=14, fontweight="bold")
    _apply_axis_style(ax)
    y_max = float(plot_df["intensity"].max()) if len(plot_df) else 1.0
    ax.set_ylim(0, y_max * 1.15 if y_max > 0 else 1.0)
    plt.tight_layout(); plt.show()

if "unaligned_components_summary_df" not in globals():
    raise NameError("unaligned_components_summary_df not found. Run the MS1-ion extraction cell first.")
if "ms1_ions_all_df" not in globals():
    raise NameError("ms1_ions_all_df not found. Run the MS1-ion extraction cell first.")

meta = unaligned_components_summary_df[
    unaligned_components_summary_df["component_index"] == COMPONENT_INDEX
]
if meta.empty:
    raise ValueError(
        f"component_index={COMPONENT_INDEX} not found. "
        f"Valid examples: {unaligned_components_summary_df['component_index'].head(10).tolist()}"
    )
meta = meta.iloc[0]

spec_df = get_component_ms1_spectrum_df(
    ms1_ions_all_df=ms1_ions_all_df,
    component_index=COMPONENT_INDEX,
    mz_min=MZ_MIN_PLOT, mz_max=MZ_MAX_PLOT,
)
topN_df = spec_df.head(int(TOP_N_IONS)).reset_index(drop=True)

title = (
    f"{meta.get('feature_name', '')}\n"
    f"{float(meta.get('target_mz', np.nan)):.4f} m/z\n"
    f"LC peak {int(meta.get('lc_peak_index'))} | "
    f"RT: {float(meta.get('rt_left')):.2f} - {float(meta.get('rt_right')):.2f} min | "
    f"DT: {float(meta.get('dt_left')):.2f} - {float(meta.get('dt_right')):.2f} ms\n\n"
    f"MS1 Spectrum"
)
plot_ms1_spectrum_atd_style(spec_df=spec_df, title=title, top_n=TOP_N_IONS)
print(f"Top {TOP_N_IONS} ions:")
display(topN_df)

##### (13) DEIMoS spectral deconvolution and MSP library export



##### Edit the cell below to tune parameters for spectral deconvolution

In [ ]:
# DEIMoS configuration variables are critical to the performance of the spectral deconvolution and must be carefully selected
# All variables must be defined; the default values here have only been tested on experimental Waters LC-IM-HDMSE data (acquired with a Synapt G2 XS mass spectrometer)

from pathlib import Path
import math

# Define how many top clustered MS1 candidates (ranked by intensity) to deconvolute per unaligned component
N_PRECURSOR_CANDS = 5

# Define name of output spectral library file
OUTPUT_TXT = Path("imfrag_demo.txt")

# If True, the output spectral library file will be OVERWRITTEN if a .txt file with the same name already exist in THE directory
# If False, an error will be thrown if the library file already exists
OVERWRITE_OUTPUT = True

# Dimensions and relative windows used to connect MS1 and MS2 peaks prior to scoring by drift time profiles
PAIR_DIMS = ["drift_time", "retention_time"] # Dimensions used for pairing MS1 & MS2; recommended to use both DT and RT if available
PAIR_LOW = [-0.12, -0.10] # Lower (relative) offsets from the MS1 center for each PAIR_DIMS dimension (e.g., -0.12 ms and -0.10 s for DT and RT, respectively)
PAIR_HIGH = [1.40, 0.10] # Upper (relative) offsets for each PAIR_DIMS dimension (e.g., +1.40 ms and +0.10 s for DT and RT, respectively)
PAIR_CE = 35 # Collision energy passed into the pairing model; for a ramp, use the midpoint of the values (e.g., 35 for a 25-45 V ramp)
REQUIRE_MS1_GT_MS2 = False # If True, discards MS2 peaks with intensity >= MS1; recommended to keep False in case of very high intense fragments 

# Profile extraction settings for similarity scoring
# DEIMoS extracts intensity profiles along the defined dimensiosn within windows around each MS1 target
# The resulting profiles are then compared by cosine-like similariy to yield drift_time_score (or retention_time_score if desired)
PROF_DIMS = ["mz", "drift_time", "retention_time"] # Dimensions along which profiles may be extracted; recommended to use both DT and RT if available (NOTE: This is computationally expensive)
PROF_LOW = [-200e-6, -0.1, -0.10]  # Lower (relative) windows around the MS1 center for each PROF_DIMS (e.g., -200e6 Da, -0.10 ms, and -0.10 s for m/z, DT, and RT, respectively)
PROF_HIGH = [600e-6, 0.1, 0.10] # Upper (relative) windows for each PROF_DIMS
PROF_REL = [True, True, True] # Define whether each PROF_DIMS window is treated as relative to the MS1 center
PROFILE_DIM = "drift_time" # Dimension to score along (the profile axis for similarity)
PROFILE_RES = 0.01 # Bin size for the profile along PROFILE_DIM (e.g., 0.01 drift time units)

# Raw denoising thresholds before peak picking
MS1_THRESH_RAW = 500 # Minimum raw intensity to retain MS1 data points before smoothing/peak-picking; recommended: 500-1000
MS2_THRESH_RAW = 2000 # Minimum raw intensity to retain MS2 data points; recommended: 1000-2000

# Smoothing and peak picking parameters
# Smoothing radius per dimension; number of iterations for each MS tier
# These default values appear to be working; recommended NOT to change
SMOOTH_RADIUS = {"mz": 0, "drift_time": 1, "retention_time": 0} # Smoothing radius per dimension 
SMOOTH_ITER_MS1 = 7 # Number of iterations for MS1 tier
SMOOTH_ITER_MS2 = 5 # Number of iterations for MS2 tier

# Peak-picking neighborhood radius per dimension 
# These default values appear to be working; recommended NOT to change
PK_RADIUS = {"mz": 2, "drift_time": 10, "retention_time": 0}
PK_RADIUS_MS2 = {"mz": 1, "drift_time": 8, "retention_time": 0}

# Feature-level gating after peak picking
MS1_FEAT_MIN = 1e4 # Primary minimum MS1 peak intensity to consider an MS1 target for deconvolution

# Fallback MS1 intensity threshold; if no candidates pass MS1_FEAT_MIN and this value is not None, the code will fallback to this lower threshold
# Set to None to disable fallback behavior (not recommended)
MS1_FEAT_MIN_FALLBACK = 2e3  # Fallback MS1 intensity threshold
MS2_FEAT_MIN = 2e3 # Minimum MS2 peak intensity to include in candidate MS2 peak set

# Exclude MS2 peaks whose m/z is more than this many Da above the precursor m/z
# Fragments can be assumed to have lower m/z values than MS1 feature 
MS2_MAX_MZ_DELTA = 0.0

# Profile similarity scoring
DT_SCORE_MIN = 0.8 # Minimum profile similarity (e.g., cosine-like) along PROFILE_DIM to keep a putative pair; recommended: 0.8
TOP_N = 2 # For each MS1 target, export at most the top N MS1 groups after scoring and aggregation

# Mass tolerances
MZ_TOL_PPM = 50 # Mass tolerance (ppm) for selecting MS1 targets near the user-defined precursor m/z
MZ_PAIR_ABS = 0.01 # Absolute fallback window for m/z-only pairing when PAIR_DIMS don’t intersect

# Printing and debugging 
VERBOSE = True # Recommended to keep True for tracking and debugging 

##### Run the cell below to deconvolve LC-IM-HDMSE spectra and write an MSP library


In [ ]:
from pathlib import Path
import math
import traceback
import re

import numpy as np
import pandas as pd
from scipy.spatial import KDTree

def _patch_deimos_for_pandas3(deimos_mod) -> None:
    cls = deimos_mod.deconvolution.MS2Deconvolution
    if getattr(cls, "_imfrag_pandas3_patched", False):
        return

    def construct_putative_pairs(self,
                                 dims=("drift_time", "retention_time"),
                                 low=(-0.12, -0.1),
                                 high=(1.4, 0.1),
                                 ce=None,
                                 model=deimos_mod.deconvolution.offset_correction_model,
                                 require_ms1_greater_than_ms2=True,
                                 error_tolerance=0.12):
        dims = deimos_mod.utils.safelist(dims)
        low = deimos_mod.utils.safelist(low)
        high = deimos_mod.utils.safelist(high)
        deimos_mod.utils.check_length([dims, low, high])
        if ce is None:
            raise ValueError("Collision energy must be specified.")
        v_ms1 = self.ms1_features[dims].to_numpy(copy=True)
        v_ms2 = self.ms2_features[dims].to_numpy(copy=True)

        for i, _dim in enumerate(dims):
            tol = (high[i] - low[i]) / 2
            v_ms2[:, i] = v_ms2[:, i] + tol + low[i]
            v_ms1[:, i] = v_ms1[:, i] / tol
            v_ms2[:, i] = v_ms2[:, i] / tol

        ms1_tree = KDTree(v_ms1)
        ms2_tree = KDTree(v_ms2)
        sdm = ms1_tree.sparse_distance_matrix(ms2_tree, 1, p=np.inf, output_type="coo_matrix")
        ms1_matches = self.ms1_features.loc[sdm.row, :].reset_index()
        ms2_matches = self.ms2_features.loc[sdm.col, :].reset_index()
        if "drift_time" in dims:
            ms2_matches["drift_time_raw"] = ms2_matches["drift_time"].to_numpy(copy=True)
            ms2_matches["drift_time"] = model(
                ms2_matches["drift_time"].to_numpy(copy=True),
                ms2_matches["mz"].to_numpy(copy=True),
                ms1_matches["mz"].to_numpy(copy=True),
                ce=ce,
            )
        ms1_matches.columns = [c + "_ms1" for c in ms1_matches.columns]
        ms2_matches.columns = [c + "_ms2" for c in ms2_matches.columns]
        decon_pairs = pd.concat((ms1_matches, ms2_matches), axis=1)
        if "drift_time" in dims:
            decon_pairs["drift_time_error"] = np.abs(decon_pairs["drift_time_ms1"] - decon_pairs["drift_time_ms2"])
        if require_ms1_greater_than_ms2:
            decon_pairs = decon_pairs.loc[decon_pairs["intensity_ms1"] >= decon_pairs["intensity_ms2"], :]
        if "drift_time" in dims:
            decon_pairs = decon_pairs.loc[decon_pairs["drift_time_error"] <= error_tolerance, :]
        decon_pairs = decon_pairs.groupby(by="index_ms1", as_index=False).filter(lambda x: len(x) > 0)
        decon_pairs = decon_pairs.sort_values(by=["index_ms1", "index_ms2"]).reset_index(drop=True)
        self.decon_pairs = decon_pairs
        return self.decon_pairs

    cls.construct_putative_pairs = construct_putative_pairs
    cls._imfrag_pandas3_patched = True

_patch_deimos_for_pandas3(deimos)

def _coerce_path(x):
    if x is None:
        return None
    return x if isinstance(x, Path) else Path(str(x))


required_cluster_cols = [
    "component_index", "cluster_id", "mz_center", "intensity_sum", "n_peaks", "mz_min", "mz_max",
    "file_key", "row_index", "lc_peak_index", "feature_name", "target_mz",
    "rt_left", "rt_right", "dt_left", "dt_right", "mu_ms2_to_ms1", "FWHM_ms2",
]
missing = [c for c in required_cluster_cols if c not in clustered_ms1_all_df.columns]
if missing:
    raise ValueError(f"clustered_ms1_all_df is missing required columns: {missing}")

if "H5_DIR" not in globals() or H5_DIR is None:
    raise ValueError("H5_DIR is not defined.")
H5_DIR = _coerce_path(H5_DIR)
if not H5_DIR.exists():
    raise FileNotFoundError(f"H5_DIR does not exist: {H5_DIR}")

INPUT_SHEET = _coerce_path(globals().get("TARGETS_XLSX", None))
if INPUT_SHEET is not None and not INPUT_SHEET.exists():
    print(f"WARNING: TARGETS_XLSX was set but does not exist: {INPUT_SHEET}. Proceeding without metadata.")

TOP_N_CANDS_PER_COMPONENT = int(N_PRECURSOR_CANDS)
DECONVOLVE_TARGET_MZ_TOO = True
DEDUP_PPM_FOR_TARGET_LIST = 10

OUTPUT_TXT = _coerce_path(OUTPUT_TXT)
OVERWRITE_OUTPUT = bool(OVERWRITE_OUTPUT)
VERBOSE = bool(VERBOSE)

def _ppm_abs(mz, ppm):
    return float(mz) * float(ppm) * 1e-6

def _mz_close(a, b, ppm):
    try:
        a = float(a); b = float(b)
        return abs(a - b) <= _ppm_abs(b, ppm)
    except Exception:
        return False

def _apply_alias_renames(df, aliases):
    rename = {alias: canon for alias, canon in aliases.items()
              if alias in df.columns and canon not in df.columns}
    return df.rename(columns=rename) if rename else df

def _align_by_keys(preferred, *parallel_lists, keys_set=None):
    if keys_set is None:
        keys_set = set(preferred)
    keep = [k for k in preferred if k in keys_set]
    idxs = [preferred.index(k) for k in keep]
    out = [keep] + [[L[i] for i in idxs] for L in parallel_lists]
    return tuple(out)

def _radius_for(dims, radius_map):
    return [radius_map[d] for d in dims]

def select_ms1_targets_by_mz(ms1_pk, mz_list, mz_tol_ppm=10):
    if not mz_list:
        return ms1_pk.iloc[0:0].copy()
    if "mz" not in ms1_pk.columns:
        raise ValueError("ms1_pk must contain an 'mz' column.")
    chosen = []
    for mz in mz_list:
        mz_abs = float(mz) * float(mz_tol_ppm) * 1e-6
        cand = ms1_pk.loc[ms1_pk["mz"].between(mz - mz_abs, mz + mz_abs)]
        if not cand.empty:
            chosen.append(cand)
    return pd.concat(chosen, ignore_index=False) if chosen else ms1_pk.iloc[0:0].copy()

def _build_peak_tables(ms1_raw, ms2_raw):
    def _pipe(raw, threshold, smooth_iter, pk_radius_map):
        df = raw.copy()
        df = deimos.threshold(df, threshold=float(threshold))
        if df.empty:
            return df
        factors = deimos.build_factors(df, dims="detect")
        index = deimos.build_index(df, factors)
        index_keys = set(getattr(index, "keys", lambda: [])())
        smooth_dims_pref = ["mz", "drift_time", "retention_time"]
        smooth_dims, = _align_by_keys(smooth_dims_pref, keys_set=index_keys)
        smooth_radius = _radius_for(smooth_dims, SMOOTH_RADIUS)
        df = deimos.filters.smooth(
            df, index=index, dims=smooth_dims,
            radius=smooth_radius, iterations=smooth_iter,
        )
        if df.empty:
            return df
        factors = deimos.build_factors(df, dims=smooth_dims)
        peaks = deimos.peakpick.persistent_homology(
            df, factors=factors, dims=smooth_dims,
            radius=_radius_for(smooth_dims, pk_radius_map),
        )
        if peaks.empty:
            return peaks
        return peaks.sort_values("persistence", ascending=False).reset_index(drop=True)

    ms1_pk = _pipe(ms1_raw, MS1_THRESH_RAW, SMOOTH_ITER_MS1, PK_RADIUS)
    ms2_pk = _pipe(ms2_raw, MS2_THRESH_RAW, SMOOTH_ITER_MS2, {**PK_RADIUS, **PK_RADIUS_MS2})
    return ms1_pk, ms2_pk

_h5_cache: dict[str, tuple] = {}

def load_h5_once(h5_path: Path):
    key = str(Path(h5_path).resolve())
    if key in _h5_cache:
        return _h5_cache[key]

    ms1_raw = deimos.load(str(h5_path), key="ms1",
                          columns=["mz", "drift_time", "retention_time", "intensity"])
    ms2_raw = deimos.load(str(h5_path), key="ms2",
                          columns=["mz", "drift_time", "retention_time", "intensity"])

    aliases = {
        "rt": "retention_time", "scan_time": "retention_time", "time": "retention_time",
        "mobility": "drift_time", "dt": "drift_time", "drift": "drift_time",
    }
    ms1_raw = _apply_alias_renames(ms1_raw, aliases)
    ms2_raw = _apply_alias_renames(ms2_raw, aliases)

    ms1_pk, ms2_pk = _build_peak_tables(ms1_raw, ms2_raw)
    if VERBOSE:
        print(f"h5 file: {Path(h5_path).name} | MS1 peaks: {len(ms1_pk):,} | MS2 peaks: {len(ms2_pk):,}")
    _h5_cache[key] = (ms1_raw, ms2_raw, ms1_pk, ms2_pk)
    return _h5_cache[key]

# Deconvolve one target via DEIMoS MS2Deconvolution
def deconvolve_one_target(ms1_raw, ms2_raw, ms1_pk, ms2_pk, target_mz, tag="", src_name=""):
    if VERBOSE:
        print(f"  Deconvolving target: {tag}")

    ms1_targets_all = select_ms1_targets_by_mz(ms1_pk, [float(target_mz)], mz_tol_ppm=MZ_TOL_PPM)
    if "intensity" in ms1_targets_all.columns:
        ms1_targets = ms1_targets_all.loc[ms1_targets_all["intensity"] >= MS1_FEAT_MIN].copy()
    else:
        ms1_targets = ms1_targets_all.copy()

    if len(ms1_targets) == 0 and ("intensity" in ms1_targets_all.columns) and (MS1_FEAT_MIN_FALLBACK is not None):
        if VERBOSE:
            print(f"    No MS1 targets at >= {MS1_FEAT_MIN:.0f}; fallback >= {MS1_FEAT_MIN_FALLBACK:.0f}")
        ms1_targets = ms1_targets_all.loc[ms1_targets_all["intensity"] >= MS1_FEAT_MIN_FALLBACK].copy()

    if len(ms1_targets) == 0:
        return []
    ms1_targets = ms1_targets.reset_index(drop=True)

    ms2_peaks_deconv = deimos.threshold(ms2_pk, threshold=MS2_FEAT_MIN)
    if len(ms2_peaks_deconv) == 0:
        return []
    if "mz" in ms2_peaks_deconv.columns and MS2_MAX_MZ_DELTA is not None:
        prec_mz = float(ms1_targets.iloc[0]["mz"])
        cutoff = prec_mz + float(MS2_MAX_MZ_DELTA)
        ms2_peaks_deconv = ms2_peaks_deconv.loc[ms2_peaks_deconv["mz"] <= cutoff].copy()
    ms2_peaks_deconv = ms2_peaks_deconv.reset_index(drop=True)
    if len(ms2_peaks_deconv) == 0:
        return []

    decon = deimos.deconvolution.MS2Deconvolution(ms1_targets, ms1_raw, ms2_peaks_deconv, ms2_raw)

    pair_keys = set(ms1_raw.columns).intersection(set(ms2_raw.columns))
    pair_dims, pair_low, pair_high = _align_by_keys(PAIR_DIMS, PAIR_LOW, PAIR_HIGH, keys_set=pair_keys)
    if not pair_dims:
        pair_dims = ["mz"]
        pair_low = [-MZ_PAIR_ABS]
        pair_high = [MZ_PAIR_ABS]
        if VERBOSE:
            print("    Pairing fallback: m/z-only")

    decon.construct_putative_pairs(
        dims=pair_dims, low=pair_low, high=pair_high,
        ce=PAIR_CE, model=deimos.deconvolution.offset_correction_model,
        require_ms1_greater_than_ms2=REQUIRE_MS1_GT_MS2,
        error_tolerance=0.3,
    )

    prof_keys = set(ms1_raw.columns)
    prof_dims, prof_low, prof_high, prof_rel = _align_by_keys(
        PROF_DIMS, PROF_LOW, PROF_HIGH, PROF_REL, keys_set=prof_keys
    )
    if not prof_dims:
        return []
    decon.configure_profile_extraction(dims=prof_dims, low=prof_low, high=prof_high, relative=prof_rel)
    profile_dim = PROFILE_DIM if PROFILE_DIM in prof_dims else prof_dims[0]
    res = decon.apply(dims=profile_dim, resolution=PROFILE_RES)

    score_col = f"{profile_dim}_score"
    if score_col not in res.columns:
        return []
    res = res.dropna(subset=[score_col]).copy()
    kept = res.loc[res[score_col] >= DT_SCORE_MIN].copy()
    if len(kept) == 0:
        return []

    ms1_cols = [c for c in kept.columns if c.endswith("_ms1")]
    ms1_group_keys = [c for c in ms1_cols if c != "persistence_ms1"]
    agg_map = {"persistence_ms1": "max", score_col: list}
    if "mz_ms2" in kept.columns:
        agg_map["mz_ms2"] = list
    if "intensity_ms2" in kept.columns:
        agg_map["intensity_ms2"] = list

    grouped = (
        kept.groupby(ms1_group_keys, as_index=False, dropna=False)
            .agg(agg_map)
            .sort_values(by="persistence_ms1", ascending=False)
            .reset_index(drop=True)
    )
    grouped_top = grouped.head(TOP_N).copy()

    cands = []
    for _, row in grouped_top.iterrows():
        mz_list = row.get("mz_ms2", [])
        I_list = row.get("intensity_ms2", [])
        if isinstance(mz_list, list) and len(mz_list) and isinstance(mz_list[0], list):
            mz_list = [m for sub in mz_list for m in sub]
        if isinstance(I_list, list) and len(I_list) and isinstance(I_list[0], list):
            I_list = [v for sub in I_list for v in sub]
        cands.append({
            "prec_mz": float(row.get("mz_ms1", np.nan)),
            "prec_int": float(row.get("intensity_ms1", np.nan)),
            "mz_ms2": mz_list if isinstance(mz_list, list) else [],
            "intensity_ms2": I_list if isinstance(I_list, list) else [],
            "scores": row.get(score_col, []) if isinstance(row.get(score_col, []), list) else [],
        })
    return cands

# Write MSP file
def _fmt_mz(x):
    try:
        return f"{float(x):.4f}"
    except Exception:
        return str(x)

def _fmt_int(x):
    if pd.isna(x):
        return "0"
    try:
        return str(int(round(float(x))))
    except Exception:
        return str(x)

def write_entry_blocks(txt_fp, base_meta: dict, target_mz: float, cand_list: list):
    feat_name = str(base_meta.get("Feature Name", "")).strip()
    adduct = str(base_meta.get("Adduct", "")).strip()
    formula = str(base_meta.get("Molecular Formula", "")).strip()

    if len(cand_list) == 0:
        txt_fp.write(f"NAME: {feat_name}\n")
        txt_fp.write(f"PRECURSORMZ: {_fmt_mz(target_mz)}\n")
        txt_fp.write(f"IONMODE: {adduct}\n")
        txt_fp.write(f"FORMULA: {formula}\n")
        txt_fp.write("RETENTIONTIMEMINS:\n")
        txt_fp.write("CCS_SQA:\n")
        txt_fp.write("Num Peaks: 0\n\n")
        return

    multi = (len(cand_list) > 1)
    for i, cand in enumerate(cand_list, 1):
        name_line = feat_name if not multi else f"{feat_name} (cand {i})"

        # Fragments only
        peaks_mz = list(cand.get("mz_ms2", []) or [])
        peaks_I = list(cand.get("intensity_ms2", []) or [])
        n = min(len(peaks_mz), len(peaks_I))
        peaks_mz = peaks_mz[:n]
        peaks_I = peaks_I[:n]

        txt_fp.write(f"NAME: {name_line}\n")
        txt_fp.write(f"PRECURSORMZ: {_fmt_mz(target_mz)}\n")
        txt_fp.write(f"IONMODE: {adduct}\n")
        txt_fp.write(f"FORMULA: {formula}\n")
        txt_fp.write("RETENTIONTIMEMINS:\n")
        txt_fp.write("CCS_SQA:\n")

        rounded_mz = [_fmt_mz(m) for m in peaks_mz]
        dedup: dict[str, float] = {}
        for mz_str, inten in zip(rounded_mz, peaks_I):
            try:
                inten_val = float(inten)
            except Exception:
                inten_val = 0.0
            if mz_str not in dedup or inten_val > dedup[mz_str]:
                dedup[mz_str] = inten_val
        mz_sorted = sorted(dedup.keys(), key=lambda x: float(x))
        int_sorted = [dedup[mz] for mz in mz_sorted]

        txt_fp.write(f"Num Peaks: {len(mz_sorted)}\n")
        for mz_str, inten_val in zip(mz_sorted, int_sorted):
            txt_fp.write(f"{mz_str} {_fmt_int(inten_val)}\n")
        txt_fp.write("\n")

# Optional metadata pull from TARGETS_XLSX
meta_df = None
if INPUT_SHEET is not None and INPUT_SHEET.exists():
    try:
        meta_df = pd.read_excel(INPUT_SHEET)
    except Exception as e:
        print(f"WARNING: could not read INPUT_SHEET: {e}")
        meta_df = None

def _lookup_meta(feature_name, target_mz, file_key):
    base = {"Adduct": "", "Ion Mode": "", "Molecular Formula": ""}
    if meta_df is None or meta_df.empty:
        return base
    cols = set(meta_df.columns)
    fn_col = "Feature Name" if "Feature Name" in cols else None
    mz_col = "Target m/z" if "Target m/z" in cols else None
    file_col = "File Name" if "File Name" in cols else None
    formula_col = next((c for c in ["Molecular Formula", "Molecular Fomula", "Formula"] if c in cols), None)
    adduct_col = "Adduct" if "Adduct" in cols else None
    ion_col = "Ion Mode" if "Ion Mode" in cols else None
    df = meta_df

    try:
        m = pd.Series([True] * len(df))
        if fn_col:
            m &= (df[fn_col].astype(str).str.strip() == str(feature_name).strip())
        if mz_col:
            m &= (df[mz_col].astype(float) == float(target_mz))
        if file_col:
            m &= (df[file_col].astype(str).str.strip() == str(file_key).strip())
        hit = df.loc[m]
        if hit.empty:
            m = pd.Series([True] * len(df))
            if fn_col:
                m &= (df[fn_col].astype(str).str.strip() == str(feature_name).strip())
            if file_col:
                m &= (df[file_col].astype(str).str.strip() == str(file_key).strip())
            hit = df.loc[m]
        if hit.empty:
            return base
        r = hit.iloc[0]
        if adduct_col:
            base["Adduct"] = str(r.get(adduct_col, "")).strip()
        if ion_col:
            base["Ion Mode"] = str(r.get(ion_col, "")).strip()
        if formula_col:
            base["Molecular Formula"] = str(r.get(formula_col, "")).strip()
        return base
    except Exception:
        return base

def build_component_target_list(comp_df: pd.DataFrame, top_n: int, include_target_mz: bool = True):
    top = comp_df.sort_values("intensity_sum", ascending=False).head(int(top_n)).copy()
    targets = []
    for i, r in enumerate(top.itertuples(index=False), 1):
        mzv = float(getattr(r, "mz_center"))
        targets.append({"kind": "cand", "mz": mzv, "label": f"cand{i}_mz{mzv:.4f}"})
    if include_target_mz:
        tmz = float(comp_df["target_mz"].iloc[0])
        targets.append({"kind": "target", "mz": tmz, "label": f"target_mz{tmz:.4f}"})
    deduped = []
    for t in targets:
        if not any(_mz_close(t["mz"], u["mz"], DEDUP_PPM_FOR_TARGET_LIST) for u in deduped):
            deduped.append(t)
    return deduped

if OUTPUT_TXT.exists() and not OVERWRITE_OUTPUT:
    raise FileExistsError(f"{OUTPUT_TXT} already exists. Set OVERWRITE_OUTPUT=True to overwrite.")

group_keys = ["file_key", "row_index", "lc_peak_index", "component_index"]
n_components = clustered_ms1_all_df[group_keys].drop_duplicates().shape[0]
print(f"Groups to process: {n_components:,}")
print(f"H5_DIR = {H5_DIR.resolve()}")
if INPUT_SHEET is not None:
    print(f"INPUT_SHEET = {INPUT_SHEET.resolve()}")

ok = skipped_missing_h5 = failed = 0

with open(OUTPUT_TXT, "w", encoding="utf-8") as fout:
    for gvals, comp_df in clustered_ms1_all_df.groupby(group_keys, dropna=False, sort=False):
        try:
            file_key, row_index, lc_peak_index, component_index = gvals
            file_key = str(file_key).strip()

            h5_path = (H5_DIR / file_key)
            if h5_path.suffix.lower() != ".h5":
                h5_path = h5_path.with_suffix(".h5")

            if not h5_path.exists():
                skipped_missing_h5 += 1
                if VERBOSE:
                    print(f"\n[SKIP] missing h5: {h5_path.name} (component_index={component_index}, lc_peak_index={lc_peak_index})")
                targets = build_component_target_list(comp_df, TOP_N_CANDS_PER_COMPONENT, include_target_mz=DECONVOLVE_TARGET_MZ_TOO)
                for t in targets:
                    base_meta = {"Feature Name": f"{comp_df['feature_name'].iloc[0]}__comp{component_index}__lc{lc_peak_index}__{t['label']}"}
                    base_meta.update(_lookup_meta(comp_df["feature_name"].iloc[0], comp_df["target_mz"].iloc[0], file_key))
                    write_entry_blocks(fout, base_meta, t["mz"], cand_list=[])
                continue

            if VERBOSE:
                feat = str(comp_df["feature_name"].iloc[0])
                tmz = float(comp_df["target_mz"].iloc[0])
                print(f"{feat} | m/z: {tmz:.4f} | {h5_path.name}")

            ms1_raw, ms2_raw, ms1_pk, ms2_pk = load_h5_once(h5_path)

            targets = build_component_target_list(
                comp_df=comp_df,
                top_n=TOP_N_CANDS_PER_COMPONENT,
                include_target_mz=DECONVOLVE_TARGET_MZ_TOO,
            )

            for t in targets:
                deconv_mz = float(t["mz"])
                feat = str(comp_df["feature_name"].iloc[0]).strip()
                if t["kind"] == "cand":
                    m = re.search(r"cand(\d+)", t["label"])
                    cand_num = m.group(1) if m else "?"
                    printed = f"{feat}_target | m/z: {deconv_mz:.4f}"
                    name_short = f"{feat}_Cand{cand_num}"
                else:
                    printed = f"{feat}_target | m/z: {deconv_mz:.4f}"
                    name_short = f"{feat}_Target"

                base_meta = {"Feature Name": name_short}
                base_meta.update(_lookup_meta(feat, float(comp_df["target_mz"].iloc[0]), file_key))

                cands = deconvolve_one_target(
                    ms1_raw, ms2_raw, ms1_pk, ms2_pk,
                    target_mz=deconv_mz, tag=printed, src_name=h5_path.stem,
                )
                write_entry_blocks(fout, base_meta, deconv_mz, cands)
                if VERBOSE:
                    print(f"    → Wrote {len(cands)} entry(s)")

            ok += 1

        except Exception as e:
            failed += 1
            print(f"\n[FAIL] group={gvals} error={e}")
            traceback.print_exc(limit=2)
            try:
                targets = build_component_target_list(comp_df, TOP_N_CANDS_PER_COMPONENT, include_target_mz=DECONVOLVE_TARGET_MZ_TOO)
                for t in targets:
                    base_meta = {"Feature Name": f"{comp_df['feature_name'].iloc[0]}__comp{component_index}__lc{lc_peak_index}__{t['label']}"}
                    base_meta.update(_lookup_meta(comp_df["feature_name"].iloc[0], comp_df["target_mz"].iloc[0], str(comp_df["file_key"].iloc[0])))
                    write_entry_blocks(fout, base_meta, float(t["mz"]), cand_list=[])
            except Exception:
                pass

print("\nDeconvolution complete.")
print(f"  OUTPUT_TXT: {OUTPUT_TXT.resolve()}")
print(f"  Component groups OK: {ok:,}")
print(f"  Skipped (missing h5): {skipped_missing_h5:,}")
print(f"  Failed: {failed:,}")